# 🛸 FPV Target Lock: YOLO26-nano Training, Edge Optimization & Benchmarking Pipeline

> **Autonomous Edge AI Target Detection & Tracking System**  
> *Non-Commercial Research & Educational Showcase*

This notebook documents the cloud training workflow, end-to-end NMS-free ablation experiments, TensorRT PTQ compilation, and rigorous FP32 vs. FP16 benchmarks on a simulated FPV flight testbed.

---

### Pipeline Roadmap:
1. **Environment Setup & Dataset Ingestion** — Package preparation, Hugging Face dataset detection (`dataset_v25_640`).
2. **Dynamic Dataset Manifest** — Automated YOLO dataset configuration mapping single-class `military_object`.
3. **Cloud Synchronization & Checkpoint Callbacks** — Hugging Face Hub cloud backup and run archiving.
4. **Architectural Experiments (1–4)** — Classic NMS baseline, End-to-End One-to-One head, MuSGD dual-gradient optimizer, and small target resolution preservation (`scale=0.0`, `shear=0.0`).
5. **Detection Diagnostics & Error Analysis** — False positive/negative identification and standoff aspect ratio profiling.
6. **Edge Acceleration (PTQ & QAT Pipeline)** — TensorRT INT8 compilation with automatic FP16 Fallback, Quantization-Aware Training, and ONNX export.
7. **Rigorous FP32 vs FP16 Comparative Validation (Test Split: 759 Frames)** — Performance benchmarks on an isolated flight testbed measuring latency, throughput (FPS), mAP50, and memory footprint on NVIDIA Tesla T4.


## 1. Environment Setup & Cloud Ingestion
Preparation of runtime dependencies, Hugging Face API authentication, and automated detection of the decimated dataset (`dataset_v25_640`).

In [1]:
# Клітинка 1: Встановлення необхідних пакетів у тихому режимі
import sys
import subprocess

def install_dependencies():
    print("Оновлення інструментів та встановлення базових залежностей...")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "ultralytics", "huggingface_hub", "PyYAML", "tqdm", "pandas", "opencv-python", "scipy", "matplotlib"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print("[Успішно] Бібліотеки середовища підготовлено.")
    except Exception as e:
        print(f"[Попередження] Виникла помилка при встановленні пакетів: {e}")

install_dependencies()

Оновлення інструментів та встановлення базових залежностей...
[Успішно] Бібліотеки середовища підготовлено.


In [2]:
# Клітинка 2: Авторизація на Hugging Face та адаптивна підготовка робочих шляхів
import os
import shutil
import glob
from huggingface_hub import login as hf_login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Безпечне зчитування токенів з Kaggle Secrets
try:
    HF_TOKEN = user_secrets.get_secret("HF_token")
except Exception:
    try:
        HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if HF_TOKEN:
    try:
        hf_login(token=HF_TOKEN)
        print("[Успішно] Авторизацію у Hugging Face виконано.")
    except Exception as e:
        print(f"[Попередження] Помилка авторизації Hugging Face: {e}")
else:
    print("[Попередження] Токен Hugging Face не знайдено.")

def detect_dataset_root():
    possible_versions = [
        "dataset_v25_640", 
        "dataset_v25_640_merged", 
        "dataset_v19", 
        "dataset_v18", 
        "dataset_v17", 
        "dataset_v16", 
        "dataset_v15"
    ]
    for version in possible_versions:
        possible_paths = [
            f"/kaggle/input/datasets/ituvtu/military-dataset/{version}",
            f"/kaggle/input/military-dataset/{version}",
            f"/kaggle/input/{version}",
        ]
        for path in possible_paths:
            if os.path.exists(path):
                return path, version
    for root, dirs, _ in os.walk("/kaggle/input"):
        for version in possible_versions:
            if version in dirs:
                return os.path.join(root, version), version
    raise FileNotFoundError("Не вдалося знайти директорію датасету в каталозі /kaggle/input.")

try:
    INPUT_DATASET_ROOT, DATASET_VERSION = detect_dataset_root()
    
    # Виправлення можливої подвійної вкладеності папок
    double_nested_path = os.path.join(INPUT_DATASET_ROOT, DATASET_VERSION)
    if os.path.exists(double_nested_path) and os.path.isdir(double_nested_path):
        INPUT_DATASET_ROOT = double_nested_path
        
    WORKING_DATASET_ROOT = f"/kaggle/working/{DATASET_VERSION}"
    print(f"Виявлено версію датасету: {DATASET_VERSION}")
    print(f"Робочий шлях копії: {WORKING_DATASET_ROOT}")
    
    if os.path.exists(WORKING_DATASET_ROOT):
        shutil.rmtree(WORKING_DATASET_ROOT)
    os.makedirs(WORKING_DATASET_ROOT, exist_ok=True)
    
    # --- Динамічне визначення структури оригінального датасету ---
    yolov8_style = False
    if os.path.exists(os.path.join(INPUT_DATASET_ROOT, "train", "images")):
        yolov8_style = True
        print("Виявлено структуру YOLOv8 (наприклад, train/images/)")
    else:
        print("Виявлено структуру YOLOv5 (наприклад, images/train/)")
        
    # --- Створення символьних посилань та копіювання анотацій ---
    if yolov8_style:
        for split in ["train", "val", "test"]:
            split_dir = os.path.join(WORKING_DATASET_ROOT, split)
            os.makedirs(split_dir, exist_ok=True)
            
            # Символьне посилання для картинок
            src_img_dir = os.path.join(INPUT_DATASET_ROOT, split, "images")
            dst_img_dir = os.path.join(split_dir, "images")
            if os.path.exists(src_img_dir):
                os.symlink(src_img_dir, dst_img_dir)
                
            # Копіювання виключно текстових файлів розмітки
            src_lbl_dir = os.path.join(INPUT_DATASET_ROOT, split, "labels")
            dst_lbl_dir = os.path.join(split_dir, "labels")
            if os.path.exists(src_lbl_dir):
                os.makedirs(dst_lbl_dir, exist_ok=True)
                for txt_file in glob.glob(os.path.join(src_lbl_dir, "*.txt")):
                    shutil.copy(txt_file, dst_lbl_dir)
                print(f"✅ Адаптовано та скопійовано анотації для спліту '{split}'")
    else:
        # Стандартна YOLOv5 структура
        os.symlink(os.path.join(INPUT_DATASET_ROOT, "images"), os.path.join(WORKING_DATASET_ROOT, "images"))
        os.makedirs(os.path.join(WORKING_DATASET_ROOT, "labels"), exist_ok=True)
        for split in ["train", "val", "test"]:
            src_split_dir = os.path.join(INPUT_DATASET_ROOT, "labels", split)
            dst_split_dir = os.path.join(WORKING_DATASET_ROOT, "labels", split)
            if os.path.exists(src_split_dir):
                os.makedirs(dst_split_dir, exist_ok=True)
                for txt_file in glob.glob(os.path.join(src_split_dir, "*.txt")):
                    shutil.copy(txt_file, dst_split_dir)
                print(f"✅ Скопійовано файли розмітки для спліту '{split}'")
                
except Exception as e:
    print(f"❌ Помилка під час ініціалізації робочої директорії: {e}")

[Успішно] Авторизацію у Hugging Face виконано.
Виявлено версію датасету: dataset_v25_640
Робочий шлях копії: /kaggle/working/dataset_v25_640
Виявлено структуру YOLOv8 (наприклад, train/images/)
✅ Адаптовано та скопійовано анотації для спліту 'train'
✅ Адаптовано та скопійовано анотації для спліту 'val'
✅ Адаптовано та скопійовано анотації для спліту 'test'


## 2. Dynamic Dataset Configuration
Automated generation of the YOLO dataset manifest (`data_yolov26_baseline.yaml`) mapping train, val, and test splits to single-class `military_object`.

In [3]:
# Клітинка 3: Генерація єдиного YAML-конфігу з динамічним визначенням структури
import os
import yaml

dataset_root = WORKING_DATASET_ROOT
original_yaml_path = os.path.join(INPUT_DATASET_ROOT, "data.yaml")

# Зчитуємо базову конфігурацію класів
if os.path.exists(original_yaml_path):
    with open(original_yaml_path, "r") as f:
        data_config = yaml.safe_load(f)
else:
    data_config = {"names": {0: "military"}}

# Встановлюємо правильний абсолютний шлях до робочого каталогу
data_config["path"] = dataset_root

# Перевіряємо, яка саме структура була розгорнута в Клітинці 2
if os.path.exists(os.path.join(dataset_root, "train", "images")):
    # YOLOv8 style
    data_config["train"] = "train/images"
    data_config["val"] = "val/images"
    data_config["test"] = "test/images"
else:
    # YOLOv5 style
    data_config["train"] = "images/train"
    data_config["val"] = "images/val"
    data_config["test"] = "images/test"

yaml_path_baseline = "/kaggle/working/data_yolov26_baseline.yaml"

with open(yaml_path_baseline, "w") as f:
    yaml.safe_dump(data_config, f, default_flow_style=False)

print(f"Глобальний YAML-конфіг згенеровано: {yaml_path_baseline}")
with open(yaml_path_baseline, "r") as f:
    print(f.read())

Глобальний YAML-конфіг згенеровано: /kaggle/working/data_yolov26_baseline.yaml
names:
  0: military_object
nc: 1
path: /kaggle/working/dataset_v25_640
test: test/images
train: train/images
val: val/images



## 3. Cloud Artifact Synchronization & Callbacks
Custom Hugging Face integration for real-time model checkpointing and run directory archiving.

In [4]:
# Клітинка 4: Допоміжні функції для роботи з Hugging Face, архівації та логування
import os
import shutil
import pandas as pd
from datetime import datetime
from huggingface_hub import HfApi, hf_hub_download

# 1. Створення детального текстового звіту за всіма епохами навчання
def generate_training_summary_text(run_dir):
    csv_path = os.path.join(run_dir, "results.csv")
    txt_path = os.path.join(run_dir, "training_epoch_log.txt")
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            # Очищення назв колонок від зайвих пробілів
            df.columns = df.columns.str.strip()
            
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(f"=== ПОВНИЙ ЛОГ НАВЧАННЯ ПО ЕПОХАХ ===\n")
                f.write(f"Директорія запуску: {run_dir}\n")
                f.write(f"Дата генерації звіту: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
                f.write(f"{'Епоха':<8}{'Train/Box':<12}{'Train/Cls':<12}{'Val/Box':<12}{'Val/Cls':<12}{'mAP50':<10}{'mAP50-95':<10}\n")
                f.write("-" * 80 + "\n")
                
                for _, row in df.iterrows():
                    epoch = int(row.get('epoch', 0))
                    train_box = row.get('train/box_loss', row.get('train/box', 0.0))
                    train_cls = row.get('train/cls_loss', row.get('train/cls', 0.0))
                    val_box = row.get('val/box_loss', row.get('val/box', 0.0))
                    val_cls = row.get('val/cls_loss', row.get('val/cls', 0.0))
                    map50 = row.get('metrics/mAP50(B)', row.get('metrics/mAP50', 0.0))
                    map95 = row.get('metrics/mAP50-95(B)', row.get('metrics/mAP50-95', 0.0))
                    
                    f.write(f"{epoch:<8}{train_box:<12.5f}{train_cls:<12.5f}{val_box:<12.5f}{val_cls:<12.5f}{map50:<10.5f}{map95:<10.5f}\n")
            print(f"📄 Згенеровано повний текстовий лог навчання: {txt_path}")
        except Exception as e:
            print(f"[Попередження] Не вдалося створити текстовий звіт з результатів CSV: {e}")

# 2. Архівування результатів та їх завантаження в репозиторій Hugging Face
def archive_and_upload_results(experiment_name, run_dir, repo_id, token=None):
    if not os.path.exists(run_dir) or not any(os.scandir(run_dir)):
        print(f"[Попередження] Директорія результатів {run_dir} порожня або відсутня. Архівування скасовано.")
        return
    
    # Створення текстового звіту по епохах безпосередньо перед закриттям архіву
    generate_training_summary_text(run_dir)
    
    zip_out_path = f"/kaggle/working/{experiment_name}_all_results"
    try:
        shutil.make_archive(zip_out_path, 'zip', run_dir)
        zip_file = f"{zip_out_path}.zip"
        size_mb = os.path.getsize(zip_file) / (1024 * 1024)
        print(f"✅ Створено ZIP-архів результатів навчання: {zip_file} ({size_mb:.2f} MB)")
        
        if token and token != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
            print(f"📤 Відправка архіву та найкращих ваг у репозиторій {repo_id}...")
            api = HfApi(token=token)
            api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
            
            # Завантаження повного архіву результатів
            api.upload_file(
                path_or_fileobj=zip_file,
                path_in_repo=f"{experiment_name}_all_results.zip",
                repo_id=repo_id,
                commit_message=f"Upload training results for {experiment_name}"
            )
            
            # Завантаження файлу ваг best.pt
            best_weights = os.path.join(run_dir, "weights/best.pt")
            if os.path.exists(best_weights):
                api.upload_file(
                    path_or_fileobj=best_weights,
                    path_in_repo="best.pt",
                    repo_id=repo_id,
                    commit_message=f"Upload best.pt for {experiment_name}"
                )
            print("🚀 Усі артефакти успішно збережено на Hugging Face Hub.")
    except Exception as e:
        print(f"❌ Сталася помилка під час архівування чи відправки результатів: {e}")

# 3. Вибір джерела моделі (локально або завантаження з HF)
def resolve_model_weights(experiment_name, repo_id, token=None):
    local_path = f"/kaggle/working/runs/detect/{experiment_name}/weights/best.pt"
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        print(f"✅ Знайдено локальні ваги за шляхом: {local_path}")
        return local_path
    
    print(f"📥 Локальних ваг не виявлено. Початок імпорту з Hugging Face ({repo_id})...")
    try:
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        downloaded_cache_path = hf_hub_download(
            repo_id=repo_id,
            filename="best.pt",
            token=token,
            local_dir=os.path.dirname(local_path),
            local_dir_use_symlinks=False
        )
        target_path = os.path.join(os.path.dirname(local_path), "best.pt")
        if downloaded_cache_path != target_path:
            shutil.copy(downloaded_cache_path, target_path)
        print(f"✅ Модель успішно імпортовано з HF: {target_path}")
        return target_path
    except Exception as e:
        print(f"❌ Помилка завантаження 'best.pt' з Hugging Face: {e}")
        try:
            print("Спроба завантаження резервного чекпоінту 'last.pt'...")
            downloaded_cache_path = hf_hub_download(
                repo_id=repo_id,
                filename="last.pt",
                token=token,
                local_dir=os.path.dirname(local_path),
                local_dir_use_symlinks=False
            )
            target_path = os.path.join(os.path.dirname(local_path), "best.pt")
            shutil.copy(downloaded_cache_path, target_path)
            print(f"✅ Резервні ваги завантажено та збережено як 'best.pt'")
            return target_path
        except Exception as err_last:
            print(f"❌ Не вдалося завантажити жоден файл ваг: {err_last}")
            return None

# 4. Кастомний зворотний виклик для моніторингу метрик у консолі
def create_hf_sync_callback(repo_id=None, token=None):
    api = None
    best_epoch_num = 1
    max_fitness_tracked = -1.0
    patience_counter = 0
    
    if token and repo_id:
        try:
            api = HfApi(token=token)
            api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
        except Exception:
            pass
            
    def on_fit_epoch_end(trainer):
        nonlocal best_epoch_num, max_fitness_tracked, patience_counter
        epoch_num = trainer.epoch + 1
        total_epochs = trainer.epochs
        
        loss_dict = {}
        loss_items = getattr(trainer, 'loss_items', [])
        loss_names = getattr(trainer, 'loss_names', [])
        if loss_items is not None and len(loss_items) > 0:
            try:
                losses = loss_items.tolist() if hasattr(loss_items, 'tolist') else list(loss_items)
                for name, val in zip(loss_names, losses):
                    loss_dict[name] = float(val)
            except Exception:
                pass

        metrics = getattr(trainer, 'metrics', {}) or {}
        val_precision = metrics.get('metrics/precision(B)', metrics.get('metrics/precision', 0.0))
        val_recall = metrics.get('metrics/recall(B)', metrics.get('metrics/recall', 0.0))
        val_map50 = metrics.get('metrics/mAP50(B)', metrics.get('metrics/mAP50', 0.0))
        val_map95 = metrics.get('metrics/mAP50-95(B)', metrics.get('metrics/mAP50-95', 0.0))

        current_fitness = getattr(trainer, 'fitness', 0.0) or 0.0
        args = getattr(trainer, 'args', None)
        patience_limit = int(getattr(args, 'patience', 0) or 0)
        
        if current_fitness > max_fitness_tracked:
            max_fitness_tracked = current_fitness
            best_epoch_num = epoch_num
            patience_counter = 0
            is_best_local = True
        else:
            patience_counter += 1
            is_best_local = False

        print("\n" + "="*70)
        print(f"ЕПОХА {epoch_num}/{total_epochs} | Моніторинг процесу навчання")
        print("-" * 70)
        if loss_dict:
            loss_str = " | ".join([f"{k}: {v:.4f}" for k, v in loss_dict.items()])
            print(f"📊 Втрати на навчанні: {loss_str}")
        print(f"📈 Метрики на валідації (Split: val): Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | mAP50: {val_map50:.4f} | mAP50-95: {val_map95:.4f}")
        print("-" * 70)
        if is_best_local:
            print(f"🏆 Оновлено найкращий результат (fitness): {current_fitness:.4f} (Епоха {best_epoch_num})")
        else:
            print(f"Поточний fitness: {current_fitness:.4f} | Найкращий: {max_fitness_tracked:.4f} на епосі {best_epoch_num}")
        if patience_limit > 0:
            print(f"⏱️ Стан ранньої зупинки (Patience): {patience_counter} / {patience_limit}")
        print("="*70 + "\n")
        
        # Резервне збереження
        if api and getattr(trainer, 'args', None) and getattr(trainer.args, 'rank', -1) in [0, -1]:
            last_path = str(trainer.last)
            best_path = str(trainer.best)
            if os.path.exists(last_path) and epoch_num % 5 == 0:
                try:
                    api.upload_file(
                        path_or_fileobj=last_path, path_in_repo="last.pt",
                        repo_id=repo_id, repo_type="model",
                        commit_message=f"Epoch {epoch_num}: backup of last.pt"
                    )
                except Exception:
                    pass
            if is_best_local and os.path.exists(best_path):
                try:
                    api.upload_file(
                        path_or_fileobj=best_path, path_in_repo="best.pt",
                        repo_id=repo_id, repo_type="model",
                        commit_message=f"Epoch {epoch_num}: new best.pt"
                    )
                except Exception:
                    pass
                    
    return on_fit_epoch_end

## 4. Systematic Model Training Experiments
Ablation study across 4 configurations to eliminate NMS inference latency and optimize gradient dynamics for low-SWaP FPV deployment.
- **Exp 1:** Baseline with classic NMS post-processing
- **Exp 2:** End-to-End NMS-Free (One-to-One dual-head regression)
- **Exp 3:** NMS-Free + MuSGD optimizer for dual-gradient stabilization
- **Exp 4:** NMS-Free + MuSGD + Small Target Spatial Resolution Preservation (`scale=0.0`, `shear=0.0`, `flipud=0.5`)

In [ ]:
# Клітинка 5: Експеримент 1 (Навчання моделі з використанням класичного NMS)
import torch
from ultralytics import YOLO

experiment_name_1 = "yolov26_with_nms"
repo_id_1 = f"ituvtu/{experiment_name_1.replace('_', '-')}"
run_dir_1 = f"/kaggle/working/runs/detect/{experiment_name_1}"

try:
    model_1 = YOLO("yolo26n.pt")
except Exception:
    print("[Попередження] Базовий файл yolo26n.pt не знайдено, використовується yolov8n.pt.")
    model_1 = YOLO("yolov8n.pt")

if HF_TOKEN and HF_TOKEN != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
    model_1.add_callback("on_fit_epoch_end", create_hf_sync_callback(
        repo_id=repo_id_1,
        token=HF_TOKEN
    ))

print(f"🚀 Запуск навчання Експерименту 1 ({experiment_name_1})...")
results_1 = model_1.train(
    data=yaml_path_baseline,
    epochs=100,
    imgsz=640,
    batch=-1,
    patience=20,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/kaggle/working/runs/detect',
    name=experiment_name_1,
    save=True,
    verbose=False,
    end2end=False,     # Використання традиційної NMS-детекції
    rect=True,          # Оптимізація під прямокутні кадри (640x384)
    
    # Вимкнення зайвих аугментацій для збереження дрібних об'єктів
    mosaic=0.0,
    scale=0.0,
    degrees=0.0,
    translate=0.0,
    shear=0.0,
    perspective=0.0,
    fliplr=0.0,
    flipud=0.0
)

# Архівація результатів навчання та їх відправка на HF
archive_and_upload_results(experiment_name_1, run_dir_1, repo_id_1, token=HF_TOKEN)

In [ ]:
# Клітинка 6: Експеримент 2 (Навчання End-to-End моделі без використання NMS)
import torch
from ultralytics import YOLO

experiment_name_2 = "yolov26_without_nms-aug"
repo_id_2 = f"ituvtu/{experiment_name_2.replace('_', '-')}"
run_dir_2 = f"/kaggle/working/runs/detect/{experiment_name_2}"

try:
    # Завантаження моделі з підтримкою End-to-End детекції без NMS
    model_2 = YOLO("yolo26n.pt")
    print("Використовується архітектура yolo26n.pt для End-to-End навчання.")
except Exception:
    print(Exception)

if HF_TOKEN and HF_TOKEN != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
    model_2.add_callback("on_fit_epoch_end", create_hf_sync_callback(
        repo_id=repo_id_2,
        token=HF_TOKEN
    ))

print(f"🚀 Запуск навчання Експерименту 2 ({experiment_name_2})...")
results_2 = model_2.train(
    data=yaml_path_baseline,
    epochs=100,
    imgsz=640,
    batch=-1,
    patience=25,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/kaggle/working/runs/detect',
    name=experiment_name_2,
    save=True,
    verbose=False,
    end2end=True,      # Активація режиму без NMS (End-to-End детекція)
    rect=False,
    
    # Вказані вами параметри аугментації:
    hsv_h=0.015,                      # Зміна колірного тону (Hue)
    hsv_s=0.1,                        # Зміна насиченості кольору (Saturation)
    hsv_v=0.2,                        # Зміна яскравості (Value)
    fliplr=0.5,                       # Ймовірність віддзеркалення зліва направо (50%)
    mosaic=1,                       # Ймовірність застосування мозаїки (80%)

    # Інші аугментації (можна вимкнути або залишити за замовчуванням):
    scale=0.0,                        # Рекомендовано для стійкості до різних відстаней
    translate=0.0,                    # Стійкість до зсувів об'єкта в кадрі
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0
)

# Архівація результатів навчання та їх відправка на HF
archive_and_upload_results(experiment_name_2, run_dir_2, repo_id_2, token=HF_TOKEN)

In [ ]:
# Клітинка 6: Експеримент 3 (Навчання End-to-End моделі без NMS з оптимізатором MuSGD)

import os
import yaml
import glob
import shutil
import torch
from ultralytics import YOLO
from huggingface_hub import HfApi

# 1. Систематичний неймінг для Експерименту 3
experiment_name_3 = "yolov26_musgd_e2e"
repo_id_3 = f"ituvtu/{experiment_name_3.replace('_', '-')}"
run_dir_3 = f"/kaggle/working/runs/detect/{experiment_name_3}"
yaml_path_baseline = f"/kaggle/working/data_{experiment_name_3}.yaml"

# Безпечне зчитування токенів з глобального простору
active_hf_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None

# 2. Пошук оригінального data.yaml та адаптація його кореневого шляху (path)
def generate_robust_yaml():
    original_yaml_path = None
    
    # Шукаємо перший знайдений конфігураційний файл data.yaml у вхідних даних
    for root, dirs, files in os.walk("/kaggle/input"):
        if "data.yaml" in files:
            original_yaml_path = os.path.join(root, "data.yaml")
            break
            
    if original_yaml_path:
        print(f"✅ Знайдено оригінальний конфіг: {original_yaml_path}")
        try:
            with open(original_yaml_path, "r", encoding="utf-8") as f:
                config = yaml.safe_load(f)
                
            # Встановлюємо робочий шлях 'path' як папку, що містить оригінальний data.yaml
            # Відносні шляхи сплітів (train, val, test) залишаємо без змін для збереження структури
            config["path"] = os.path.dirname(original_yaml_path)
            
            os.makedirs(os.path.dirname(yaml_path_baseline), exist_ok=True)
            with open(yaml_path_baseline, "w", encoding="utf-8") as f:
                yaml.safe_dump(config, f, default_flow_style=False)
                
            print(f"✅ Успішно записано YAML конфіг для Експерименту 3: {yaml_path_baseline}")
            print("Конфігурація згенерованого YAML-файлу:")
            print(yaml.dump(config, default_flow_style=False))
        except Exception as e:
            print(f"❌ Помилка обробки файлу конфігурації: {e}")
    else:
        print("❌ Не вдалося знайти оригінальний data.yaml у каталозі /kaggle/input.")

generate_robust_yaml()

# Запобіжна перевірка перед запуском тренування
if not os.path.exists(yaml_path_baseline):
    raise FileNotFoundError(f"❌ Не вдалося створити конфігураційний файл {yaml_path_baseline}. "
                            "Будь ласка, перевірте наявність датасету у вхідних файлах.")

# 3. Допоміжна автономна функція архівації та вивантаження результатів на HF
def archive_and_upload_results(experiment_name, run_dir, repo_id, token):
    zip_local = f"/kaggle/working/{experiment_name}_run_results"
    zip_hf_name = f"{experiment_name}_all_results.zip"
    
    if os.path.exists(run_dir):
        print(f"📦 Початок архівації результатів з папки: {run_dir}")
        try:
            shutil.make_archive(zip_local, 'zip', run_dir)
            zip_file = f"{zip_local}.zip"
            
            if os.path.exists(zip_file):
                size_mb = os.path.getsize(zip_file) / (1024 * 1024)
                print(f"✅ Локальний архів створено: {zip_file} ({size_mb:.2f} MB)")
                
                if token and token != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
                    print(f"📥 Завантаження результатів у репозиторій Hugging Face: {repo_id}...")
                    api = HfApi(token=token)
                    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
                    
                    # Вивантаження повного ZIP-архіву метрик та графіків
                    api.upload_file(
                        path_or_fileobj=zip_file,
                        path_in_repo=zip_hf_name,
                        repo_id=repo_id,
                        commit_message="Експеримент 3: Вивантаження повного архіву результатів"
                    )
                    
                    # Дублювання найкращих ваг моделі окремо у корінь репозиторію
                    best_pt = os.path.join(run_dir, "weights/best.pt")
                    if os.path.exists(best_pt):
                        api.upload_file(
                            path_or_fileobj=best_pt,
                            path_in_repo="best.pt",
                            repo_id=repo_id,
                            commit_message="Експеримент 3: Вивантаження найкращих ваг best.pt"
                        )
                    print("[Успішно] Артефакти Експерименту 3 збережено на Hugging Face.")
                else:
                    print("[Інформація] Завантаження на Hugging Face пропущено через відсутність токена.")
        except Exception as e:
            print(f"❌ Помилка під час архівації або надсилання результатів: {e}")
    else:
        print(f"⚠️ Папку результатів {run_dir} не знайдено.")

# 4. Ініціалізація та запуск навчання моделі
try:
    # Завантаження моделі з підтримкою End-to-End детекції без NMS
    model_3 = YOLO("yolo26n.pt")
    print("Використовується архітектура yolo26n.pt для End-to-End навчання.")
except Exception as e:
    print(f"[Попередження] Не вдалося завантажити yolo26n.pt, використовується yolov8n.pt: {e}")
    model_3 = YOLO("yolov8n.pt")

# Підключення зворотного виклику для синхронізації з HF
if active_hf_token and active_hf_token != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
    if 'create_hf_sync_callback' in locals() or 'create_hf_sync_callback' in globals():
        model_3.add_callback("on_fit_epoch_end", create_hf_sync_callback(
            repo_id=repo_id_3,
            token=active_hf_token
        ))
    else:
        print("[Попередження] Функцію create_hf_sync_callback не знайдено у сесії.")

print(f"🚀 Запуск навчання Експерименту 3 ({experiment_name_3})...")
results_3 = model_3.train(
    data=yaml_path_baseline,
    epochs=100,
    imgsz=640,
    batch=-1,           # Автоматичний підбір батчу під об'єм VRAM
    patience=25,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/kaggle/working/runs/detect',
    name=experiment_name_3,
    save=True,
    verbose=False,
    end2end=True,       # Режим End-to-End детекції
    optimizer="MuSGD",  # Оптимізатор MuSGD
    rect=False,
    
    # Конфігурація колірних та просторових аугментацій
    hsv_h=0.015,
    hsv_s=0.1,
    hsv_v=0.2,
    fliplr=0.5,
    mosaic=1.0,

    # Інші геометричні перетворення
    scale=0.0,
    translate=0.0,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0
)

# 5. Створення архіву результатів тренування та надсилання у репозиторій Hugging Face
archive_and_upload_results(experiment_name_3, run_dir_3, repo_id_3, token=active_hf_token)

In [ ]:
# Клітинка 6: Експеримент 4 (Навчання End-to-End моделі без NMS з оптимізатором MuSGD та оптимізованими аугментаціями)

import os
import yaml
import glob
import shutil
import torch
from ultralytics import YOLO
from huggingface_hub import HfApi

# 1. Систематичний неймінг для Експерименту 4
experiment_name_4 = "yolov26_musgd_e2e_experiment4"
repo_id_4 = f"ituvtu/{experiment_name_4.replace('_', '-')}"
run_dir_4 = f"/kaggle/working/runs/detect/{experiment_name_4}"
yaml_path_baseline = f"/kaggle/working/data_{experiment_name_4}.yaml"

# Безпечне зчитування токенів з глобального простору
active_hf_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None

# 2. Пошук оригінального data.yaml та адаптація його кореневого шляху (path)
def generate_robust_yaml():
    original_yaml_path = None
    
    # Шукаємо перший знайдений конфігураційний файл data.yaml у вхідних даних
    for root, dirs, files in os.walk("/kaggle/input"):
        if "data.yaml" in files:
            original_yaml_path = os.path.join(root, "data.yaml")
            break
            
    if original_yaml_path:
        print(f"✅ Знайдено оригінальний конфіг: {original_yaml_path}")
        try:
            with open(original_yaml_path, "r", encoding="utf-8") as f:
                config = yaml.safe_load(f)
                
            # Встановлюємо робочий шлях 'path' як папку, що містить оригінальний data.yaml
            config["path"] = os.path.dirname(original_yaml_path)
            
            os.makedirs(os.path.dirname(yaml_path_baseline), exist_ok=True)
            with open(yaml_path_baseline, "w", encoding="utf-8") as f:
                yaml.safe_dump(config, f, default_flow_style=False)
                
            print(f"✅ Успішно записано YAML конфіг для Експерименту 4: {yaml_path_baseline}")
            print("Конфігурація згенерованого YAML-файлу:")
            print(yaml.dump(config, default_flow_style=False))
        except Exception as e:
            print(f"❌ Помилка обробки файлу конфігурації: {e}")
    else:
        print("❌ Не вдалося знайти оригінальний data.yaml у каталозі /kaggle/input.")

generate_robust_yaml()

# Запобіжна перевірка перед запуском тренування
if not os.path.exists(yaml_path_baseline):
    raise FileNotFoundError(f"❌ Не вдалося створити конфігураційний файл {yaml_path_baseline}. "
                            "Будь ласка, перевірте наявність датасету у вхідних файлах.")

# 3. Допоміжна автономна функція архівації та вивантаження результатів на HF
def archive_and_upload_results(experiment_name, run_dir, repo_id, token):
    # Генерація текстового звіту по епохах, якщо функція присутня в глобальному просторі
    if 'generate_training_summary_text' in globals():
        generate_training_summary_text(run_dir)
        
    zip_local = f"/kaggle/working/{experiment_name}_run_results"
    zip_hf_name = f"{experiment_name}_all_results.zip"
    
    if os.path.exists(run_dir):
        print(f"📦 Початок архівації результатів з папки: {run_dir}")
        try:
            shutil.make_archive(zip_local, 'zip', run_dir)
            zip_file = f"{zip_local}.zip"
            
            if os.path.exists(zip_file):
                size_mb = os.path.getsize(zip_file) / (1024 * 1024)
                print(f"✅ Локальний архів створено: {zip_file} ({size_mb:.2f} MB)")
                
                if token and token != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
                    print(f"📥 Завантаження результатів у репозиторій Hugging Face: {repo_id}...")
                    api = HfApi(token=token)
                    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
                    
                    # Вивантаження повного ZIP-архіву метрик та графіків
                    api.upload_file(
                        path_or_fileobj=zip_file,
                        path_in_repo=zip_hf_name,
                        repo_id=repo_id,
                        commit_message="Експеримент 4: Вивантаження повного архіву результатів"
                    )
                    
                    # Дублювання найкращих ваг моделі окремо у корінь репозиторію
                    best_pt = os.path.join(run_dir, "weights/best.pt")
                    if os.path.exists(best_pt):
                        api.upload_file(
                            path_or_fileobj=best_pt,
                            path_in_repo="best.pt",
                            repo_id=repo_id,
                            commit_message="Експеримент 4: Вивантаження найкращих ваг best.pt"
                        )
                    print("[Успішно] Артефакти Експерименту 4 збережено на Hugging Face.")
                else:
                    print("[Інформація] Завантаження на Hugging Face пропущено через відсутність токена.")
        except Exception as e:
            print(f"❌ Помилка під час архівації або надсилання результатів: {e}")
    else:
        print(f"⚠️ Папку результатів {run_dir} не знайдено.")

# 4. Ініціалізація моделі
try:
    model_4 = YOLO("yolo26n.pt")
    print("Використовується архітектура yolo26n.pt для End-to-End навчання.")
except Exception as e:
    print(f"[Попередження] Не вдалося завантажити yolo26n.pt, використовується yolov8n.pt: {e}")
    model_4 = YOLO("yolov8n.pt")

# Підключення зворотного виклику для синхронізації з HF
if active_hf_token and active_hf_token != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
    if 'create_hf_sync_callback' in locals() or 'create_hf_sync_callback' in globals():
        model_4.add_callback("on_fit_epoch_end", create_hf_sync_callback(
            repo_id=repo_id_4,
            token=active_hf_token
        ))
    else:
        print("[Попередження] Функцію create_hf_sync_callback не знайдено у сесії.")

# 5. Запуск навчання моделі з вашими налаштуваннями аугментацій
print(f"🚀 Запуск навчання Експерименту 4 ({experiment_name_4})...")
results_4 = model_4.train(
    data=yaml_path_baseline,
    epochs=100,
    imgsz=640,
    batch=-1,           # Автоматичний підбір батчу під об'єм VRAM
    patience=25,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/kaggle/working/runs/detect',
    name=experiment_name_4,
    save=True,
    verbose=False,
    end2end=True,       # Режим End-to-End детекції (без NMS)
    optimizer="MuSGD",  # Оптимізатор MuSGD
    rect=False,
    
    # --- Скориговані колірні аугментації ---
    hsv_h=0.015,
    hsv_s=0.3,               # збільшено з 0.1 для стійкості до погоди/тіней
    hsv_v=0.3,               # збільшено з 0.2 для стійкості до експозиції
    
    # --- Геометричні аугментації (без scale) ---
    fliplr=0.5,
    flipud=0.5,              # увімкнено (0.5) для усунення ракурсної упередженості зверху
    mosaic=1.0,              # захищає від перенавчання на великих планах
    translate=0.1,           # усунення просторової упередженості по осі Y
    degrees=15.0,            # покращення точності прогнозування меж об'єктів (box_loss)
    
    # --- Вимкнені важкі деформації (зберігаємо дрібні об'єкти) ---
    scale=0.0,               # вимкнено за вимогою
    shear=0.0,
    perspective=0.0
)

# 6. Створення архіву результатів тренування та надсилання у репозиторій Hugging Face
archive_and_upload_results(experiment_name_4, run_dir_4, repo_id_4, token=active_hf_token)

## 5. Detection Diagnostics & Error Analysis
Standoff range error analysis, false-positive detection auditing, and aspect ratio distribution analysis across target geometries.

In [ ]:
# Клітинка 7: Універсальне тестування, вивід параметрів моделі (NMS), категоризація рамок та запис результатів
import os
import cv2
import glob
import shutil
import torch
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from huggingface_hub import HfApi, hf_hub_download

# ==================== ПАРАМЕТРИ ТЕСТУВАННЯ ====================
# Вкажіть назву моделі, яку хочете протестувати: "yolov26_with_nms" або "yolov26_without_nms"
SELECTED_EXPERIMENT = "yolov26_musgd_e2e_experiment4"
# ==============================================================

selected_repo_id = f"ituvtu/{SELECTED_EXPERIMENT.replace('_', '-')}"
active_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None

# Допоміжна функція для обчислення IoU (Intersection over Union) між рамками
def calculate_single_iou(box1, box2):
    x_l = max(box1[0], box2[0])
    y_t = max(box1[1], box2[1])
    x_r = min(box1[2], box2[2])
    y_b = min(box1[3], box2[3])
    
    if x_r < x_l or y_b < y_t:
        return 0.0
        
    intersection = (x_r - x_l) * (y_b - y_t)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    if union <= 0:
        return 0.0
    return intersection / float(union)

print(f"🔄 Початок підготовки моделі для експерименту: {SELECTED_EXPERIMENT}")

# Штучне створення цільового локального файлу (підміна ваг на yolov26_pytorch_fp16.pt)
try:
    target_dir = f"/kaggle/working/runs/detect/{SELECTED_EXPERIMENT}/weights"
    fake_best_path = os.path.join(target_dir, "best.pt")
    
    if not (os.path.exists(fake_best_path) and os.path.getsize(fake_best_path) > 0):
        print("🔗 Ініційовано примусове завантаження модифікації yolov26_pytorch_fp16.pt з Hugging Face...")
        os.makedirs(target_dir, exist_ok=True)
        
        downloaded_fp16 = hf_hub_download(
            repo_id=selected_repo_id,
            filename="yolov26_pytorch_fp16.pt",
            token=active_token,
            local_dir=target_dir,
            local_dir_use_symlinks=False
        )
        
        # Перейменування завантаженого файлу на ім'я, яке шукає resolve_model_weights
        os.rename(downloaded_fp16, fake_best_path)
        print("✅ Файл yolov26_pytorch_fp16.pt успішно завантажено та підготовлено як best.pt")
    else:
        print("✅ Локальний файл ваг для підміни вже існує.")
except Exception as patch_error:
    print(f"⚠️ Попередження під час спроби підміни файлу ваг: {patch_error}")

# Крок 1. Перевірка локальних ваг або завантаження з Hugging Face
resolved_weights_path = resolve_model_weights(
    experiment_name=SELECTED_EXPERIMENT,
    repo_id=selected_repo_id,
    token=active_token
)

if resolved_weights_path and os.path.exists(resolved_weights_path):
    # Крок 2. Ініціалізація моделі
    eval_model = YOLO(resolved_weights_path)
    
    # Крок 3. Звернення до моделі та вивід її параметрів (NMS, пороги, конфігурація)
    print("\n" + "="*50)
    print("      ДИАГНОСТИКА КОНФІГУРАЦІЇ ТА ПАРАМЕТРИ МОДЕЛІ")
    print("="*50)
    print(f" Завдання (Task):              {getattr(eval_model, 'task', 'N/A')}")
    print(f" Назва моделі (Model Name):    {getattr(eval_model, 'model_name', 'N/A')}")
    
    # Отримання параметрів інференсу з overrides
    model_overrides = getattr(eval_model, 'overrides', {}) or {}
    default_conf = model_overrides.get('conf', 0.25)
    default_iou = model_overrides.get('iou', 0.7)
    max_det = model_overrides.get('max_det', 300)
    img_size = model_overrides.get('imgsz', 640)
    agnostic_nms = model_overrides.get('agnostic_nms', False)
    end2end = model_overrides.get('end2end', None)
    
    print(f" Поріг впевненості (Conf Thresh): {default_conf}")
    print(f" Поріг IoU для NMS (IoU Thresh):   {default_iou}")
    print(f" Макс. детекцій (Max Detections): {max_det}")
    print(f" Розмір кадру (Image Size):       {img_size}")
    print(f" Клас-агностичний NMS (Agnostic): {agnostic_nms}")
    print(f" Без-NMS режим (End-to-End):      {end2end if end2end is not None else 'Default (False)'}")
    
    # Отримання параметрів, збережених у чекпоінті (ckpt) під час навчання
    ckpt_args = {}
    if hasattr(eval_model, 'ckpt') and isinstance(eval_model.ckpt, dict):
        ckpt_args = eval_model.ckpt.get('train_args', {}) or {}
    end2end_model = ckpt_args.get('end2end', 'N/A')
    if ckpt_args:
        print("-" * 50)
        print(" Налаштування з етапу навчання (Checkpoint Train Args):")
        print(f"   Режим End-to-End (Без NMS): {ckpt_args.get('end2end', 'N/A')}")
        print(f"   Епохи (Epochs):             {ckpt_args.get('epochs', 'N/A')}")
        print(f"   Patience (Рання зупинка):   {ckpt_args.get('patience', 'N/A')}")
        print(f"   Оптимізатор (Optimizer):    {ckpt_args.get('optimizer', 'N/A')}")
        print(f"   Прямокутне навчання (Rect): {ckpt_args.get('rect', 'N/A')}")
        print(f"   Аугментація Mosaic:         {ckpt_args.get('mosaic', 'N/A')}")
    print("="*50 + "\n")
    
    # Визначення порогів для поточного тесту
    test_conf = 0.25 if "with-nms" in SELECTED_EXPERIMENT else 0.05
    test_iou = 0.45 if "with-nms" in SELECTED_EXPERIMENT else 0.90
    print(f"⚙️ Запуск тестування з порогами: Conf={test_conf}, IoU (для NMS)={test_iou}")
    
    # Крок 4. Запуск тестування на Comic спліті (split='test')
    test_results_dir = f"/kaggle/working/runs/detect/{SELECTED_EXPERIMENT}_test_evaluation"
    test_metrics = eval_model.val(
        data=yaml_path_baseline,
        split='test',  # Оцінюємо виключно на тестовій вибірці
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else 'cpu',
        rect=True,
        project='/kaggle/working/runs/detect',
        name=f"{SELECTED_EXPERIMENT}_test_evaluation",
        end2end = end2end_model,
        exist_ok=True,
        conf=test_conf,
        iou=test_iou
    )
    
    # Крок 5. Створення детального текстового звіту точності (test_metrics_summary.txt)
    summary_file_path = os.path.join(test_results_dir, "test_metrics_summary.txt")
    
    # Пошук папок зображень та анотацій тестового спліту
    test_img_dir = os.path.join(WORKING_DATASET_ROOT, "test", "images")
    test_lbl_dir = os.path.join(WORKING_DATASET_ROOT, "test", "labels")
    if not os.path.exists(test_img_dir):
        test_img_dir = os.path.join(WORKING_DATASET_ROOT, "images", "test")
        test_lbl_dir = os.path.join(WORKING_DATASET_ROOT, "labels", "test")
        
    test_viz_dir = os.path.join(test_results_dir, "test_visualizations")
    os.makedirs(test_viz_dir, exist_ok=True)
    
    # Глобальні лічильники рамок
    total_green = 0  # Ground Truth (Істинні об'єкти)
    total_white = 0  # True Positives (Правильні передбачення)
    total_red = 0    # False Positives (Помилкові передбачення)
    
    if os.path.exists(test_img_dir):
        valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        test_images = [img for img in glob.glob(os.path.join(test_img_dir, "*.*")) 
                       if os.path.splitext(img)[1].lower() in valid_extensions]
        
        print(f"🎨 Початок аналізу та малювання {len(test_images)} тестових зображень...")
        
        for img_path in tqdm(test_images, desc="Обчислення рамок (Зелений = GT, Білий = OK, Червоний = ERR)"):
            img = cv2.imread(img_path)
            if img is None:
                continue
            h_img, w_img = img.shape[:2]
            img_stem = Path(img_path).stem
            
            # А. Зчитуємо та зберігаємо координати Ground Truth
            gt_boxes = []
            label_path = os.path.join(test_lbl_dir, f"{img_stem}.txt")
            if os.path.exists(label_path):
                with open(label_path, "r") as f_gt:
                    for line in f_gt:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            cls_id, xc, yc, wb, hb = map(float, parts)
                            x1 = int((xc - wb / 2) * w_img)
                            y1 = int((yc - hb / 2) * h_img)
                            x2 = int((xc + wb / 2) * w_img)
                            y2 = int((yc + hb / 2) * h_img)
                            gt_boxes.append({
                                'coords': [x1, y1, x2, y2],
                                'cls': int(cls_id)
                            })
                            
            # Б. Отримуємо передбачення моделі з порогами поточного тесту
            preds = eval_model.predict(
                img_path, 
                conf=test_conf, 
                iou=test_iou,
                imgsz=640, 
                verbose=False
            )[0]
            
            pred_boxes = []
            for box in preds.boxes:
                coords = box.xyxy[0].cpu().numpy().astype(int).tolist()
                conf_val = float(box.conf[0])
                cls_id = int(box.cls[0])
                pred_boxes.append({
                    'coords': coords,
                    'cls': cls_id,
                    'conf': conf_val
                })
                
            # В. Зіставлення передбачень з GT (Bipartite Greedy Matching)
            matched_gts = set()
            matched_preds = set()
            candidates = []
            
            for g_idx, gt in enumerate(gt_boxes):
                for p_idx, pr in enumerate(pred_boxes):
                    if gt['cls'] == pr['cls']:
                        iou_val = calculate_single_iou(gt['coords'], pr['coords'])
                        if iou_val >= 0.5:
                            candidates.append((g_idx, p_idx, iou_val))
                            
            candidates.sort(key=lambda x: x[2], reverse=True)
            for g_idx, p_idx, iou_val in candidates:
                if g_idx not in matched_gts and p_idx not in matched_preds:
                    matched_gts.add(g_idx)
                    matched_preds.add(p_idx)
                    
            # Г. Малювання рамок на кадрі
            # 1. Малюємо Ground Truth (Зелений колір)
            for gt in gt_boxes:
                x1, y1, x2, y2 = gt['coords']
                c_name = eval_model.names.get(gt['cls'], str(gt['cls']))
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, f"GT: {c_name}", (x1, max(15, y1 - 5)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
                total_green += 1
                
            # 2. Малюємо передбачення моделі (Білий або Червоний колір)
            for p_idx, pr in enumerate(pred_boxes):
                x1, y1, x2, y2 = pr['coords']
                c_name = eval_model.names.get(pr['cls'], str(pr['cls']))
                conf_val = pr['conf']
                
                if p_idx in matched_preds:
                    # Зараховано як правильне (Білий колір)
                    color = (255, 255, 255)
                    label = f"OK: {c_name} {conf_val:.2f}"
                    total_white += 1
                else:
                    # Помилка детекції (Червоний колір)
                    color = (0, 0, 255)
                    label = f"ERR: {c_name} {conf_val:.2f}"
                    total_red += 1
                    
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, label, (x1, min(h_img - 5, y2 + 12)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
                
            # Збереження готового кадру у папку тестування
            cv2.imwrite(os.path.join(test_viz_dir, f"visualized_{img_stem}.jpg"), img)
            
        print(f"✅ Усі анотовані зображення збережено в: {test_viz_dir}")
    else:
        print("[Попередження] Не вдалося знайти папку з тестовими зображеннями для малювання рамок.")

    # Створення текстового звіту точності (test_metrics_summary.txt)
    try:
        with open(summary_file_path, "w", encoding="utf-8") as f:
            f.write(f"=== ТЕКСТОВИЙ ЗВІТ ТОЧНОСТІ ТЕСТУВАННЯ (СПЛІТ: TEST) ===\n")
            f.write(f"Експеримент: {SELECTED_EXPERIMENT}\n")
            f.write(f"Дата оцінки: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Файл ваг: {resolved_weights_path}\n")
            f.write("-" * 60 + "\n\n")
            
            f.write("⚙️ Активні параметри інференсу моделі:\n")
            f.write(f"  Поріг Conf:                    {test_conf}\n")
            f.write(f"  Поріг IoU для NMS:             {test_iou}\n")
            f.write(f"  Режим End-to-End (Без NMS):   {end2end if end2end is not None else 'Default (False)'}\n\n")
            
            f.write("📈 Загальні метрики точності моделі (mAP):\n")
            f.write(f"  Точність (Precision):         {test_metrics.box.mp:.6f}\n")
            f.write(f"  Повнота (Recall):              {test_metrics.box.mr:.6f}\n")
            f.write(f"  mAP@50:                        {test_metrics.box.map50:.6f}\n")
            f.write(f"  mAP@50-95:                     {test_metrics.box.map:.6f}\n\n")
            
            f.write("📊 Статистика згенерованих рамок на тестовій вибірці:\n")
            f.write(f"  Кількість істинних об'єктів (Зелені / GT):          {total_green}\n")
            f.write(f"  Кількість зарахованих правильних детекцій (Білі):    {total_white}\n")
            f.write(f"  Кількість помилкових детекцій (Червоні):             {total_red}\n")
            f.write(f"  Загальна кількість передбачень моделі:              {total_white + total_red}\n\n")
            
            f.write("⏱️ Швидкість обробки зображень (в мілісекундах на кадр):\n")
            for speed_key, speed_val in test_metrics.speed.items():
                f.write(f"  {speed_key}: {speed_val:.3f} ms\n")
            f.write("\n")
            
            f.write("📊 Детальні метрики точності по класах:\n")
            class_names = eval_model.names
            ap_class_indices = getattr(test_metrics.box, 'ap_class_index', [])
            
            if len(ap_class_indices) > 0:
                for idx, class_idx in enumerate(ap_class_indices):
                    c_name = class_names.get(class_idx, f"ID_{class_idx}")
                    p_class = test_metrics.box.p[idx] if len(test_metrics.box.p) > idx else 0.0
                    r_class = test_metrics.box.r[idx] if len(test_metrics.box.r) > idx else 0.0
                    ap50_class = test_metrics.box.ap50[idx] if len(test_metrics.box.ap50) > idx else 0.0
                    ap95_class = test_metrics.box.ap[idx] if len(test_metrics.box.ap) > idx else 0.0
                    
                    f.write(f"  Клас [{class_idx}] '{c_name}':\n")
                    f.write(f"     Precision: {p_class:.6f}\n")
                    f.write(f"     Recall:    {r_class:.6f}\n")
                    f.write(f"     mAP@50:    {ap50_class:.6f}\n")
                    f.write(f"     mAP@50-95: {ap95_class:.6f}\n")
            else:
                f.write("  Дані по окремих класах відсутні.\n")
                
        print(f"📄 Згенеровано та збережено текстовий звіт: {summary_file_path}")
    except Exception as e:
        print(f"[Попередження] Помилка створення текстового звіту точності: {e}")

    # Крок 6. Консольний вивід результатів та кількості рамок
    print("\n" + "="*50)
    print(f"      РЕЗУЛЬТАТИ ТЕСТУВАННЯ НА СПЛІТІ 'TEST' ({SELECTED_EXPERIMENT})")
    print("="*50)
    print(f" Точність (Precision):         {test_metrics.box.mp:.4f}")
    print(f" Повнота (Recall):              {test_metrics.box.mr:.4f}")
    print(f" mAP@50 (mAP):                 {test_metrics.box.map50:.4f}")
    print(f" mAP@50-95:                     {test_metrics.box.map:.4f}")
    print("-" * 50)
    print(f" 🟢 Зелені рамки (Ground Truth):        {total_green}")
    print(f" ⚪ Білі рамки (Правильні / OK):        {total_white}")
    print(f" 🔴 Червоні рамки (Помилкові / ERR):     {total_red}")
    print(f" 📦 Загальна к-сть передбачень моделі:   {total_white + total_red}")
    print("="*50 + "\n")
    
    # Крок 7. Створення фінального ZIP-архіву тестування
    if os.path.exists(test_results_dir):
        archive_name = f"/kaggle/working/{SELECTED_EXPERIMENT}_test_evaluation"
        shutil.make_archive(archive_name, 'zip', test_results_dir)
        zip_file_eval = f"{archive_name}.zip"
        size_mb = os.path.getsize(zip_file_eval) / (1024 * 1024)
        print(f"✅ Створено повний ZIP-архів тестування: {zip_file_eval} ({size_mb:.2f} MB)")
        
        # Завантаження архівованих тестів на Hugging Face
        if active_token and active_token != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
            try:
                print(f"📤 Відправка архіву на Hugging Face...")
                api = HfApi(token=active_token)
                api.upload_file(
                    path_or_fileobj=zip_file_eval,
                    path_in_repo=f"{SELECTED_EXPERIMENT}_test_evaluation.zip",
                    repo_id=selected_repo_id,
                    commit_message=f"Upload evaluation with config parameters, OK (White) and ERR (Red) bounding boxes for {SELECTED_EXPERIMENT}"
                )
                print("🚀 Архів тестування успішно завантажено на Hugging Face.")
            except Exception as e:
                print(f"❌ Не вдалося відправити тест-архів на Hugging Face: {e}")
else:
    print(f"❌ Не вдалося знайти або імпортувати ваги моделі для експерименту: {SELECTED_EXPERIMENT}")

In [ ]:
# Клітинка 7: Універсальне тестування, вивід параметрів моделі (NMS), категоризація рамок та запис результатів
import os
import cv2
import glob
import shutil
import torch
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from huggingface_hub import HfApi

# ==================== ПАРАМЕТРИ ТЕСТУВАННЯ ====================
# Вкажіть назву моделі, яку хочете протестувати: "yolov26_with_nms" або "yolov26_without_nms"
SELECTED_EXPERIMENT = "yolov26_musgd_e2e_experiment4"
# ==============================================================

selected_repo_id = f"ituvtu/{SELECTED_EXPERIMENT.replace('_', '-')}"
active_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None

# Допоміжна функція для обчислення IoU (Intersection over Union) між рамками
def calculate_single_iou(box1, box2):
    x_l = max(box1[0], box2[0])
    y_t = max(box1[1], box2[1])
    x_r = min(box1[2], box2[2])
    y_b = min(box1[3], box2[3])
    
    if x_r < x_l or y_b < y_t:
        return 0.0
        
    intersection = (x_r - x_l) * (y_b - y_t)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    if union <= 0:
        return 0.0
    return intersection / float(union)

print(f"🔄 Початок підготовки моделі для експерименту: {SELECTED_EXPERIMENT}")

# Крок 1. Перевірка локальних ваг або завантаження з Hugging Face
resolved_weights_path = resolve_model_weights(
    experiment_name=SELECTED_EXPERIMENT,
    repo_id=selected_repo_id,
    token=active_token
)

if resolved_weights_path and os.path.exists(resolved_weights_path):
    # Крок 2. Ініціалізація моделі
    eval_model = YOLO(resolved_weights_path)
    
    # Крок 3. Звернення до моделі та вивід її параметрів (NMS, пороги, конфігурація)
    print("\n" + "="*50)
    print("      ДИАГНОСТИКА КОНФІГУРАЦІЇ ТА ПАРАМЕТРИ МОДЕЛІ")
    print("="*50)
    print(f" Завдання (Task):              {getattr(eval_model, 'task', 'N/A')}")
    print(f" Назва моделі (Model Name):    {getattr(eval_model, 'model_name', 'N/A')}")
    
    # Отримання параметрів інференсу з overrides
    model_overrides = getattr(eval_model, 'overrides', {}) or {}
    default_conf = model_overrides.get('conf', 0.25)
    default_iou = model_overrides.get('iou', 0.7)
    max_det = model_overrides.get('max_det', 300)
    img_size = model_overrides.get('imgsz', 640)
    agnostic_nms = model_overrides.get('agnostic_nms', False)
    end2end = model_overrides.get('end2end', None)
    
    print(f" Поріг впевненості (Conf Thresh): {default_conf}")
    print(f" Поріг IoU для NMS (IoU Thresh):   {default_iou}")
    print(f" Макс. детекцій (Max Detections): {max_det}")
    print(f" Розмір кадру (Image Size):       {img_size}")
    print(f" Клас-агностичний NMS (Agnostic): {agnostic_nms}")
    print(f" Без-NMS режим (End-to-End):      {end2end if end2end is not None else 'Default (False)'}")
    
    # Отримання параметрів, збережених у чекпоінті (ckpt) під час навчання
    ckpt_args = {}
    if hasattr(eval_model, 'ckpt') and isinstance(eval_model.ckpt, dict):
        ckpt_args = eval_model.ckpt.get('train_args', {}) or {}
    end2end_model = ckpt_args.get('end2end', 'N/A')
    if ckpt_args:
        print("-" * 50)
        print(" Налаштування з етапу навчання (Checkpoint Train Args):")
        print(f"   Режим End-to-End (Без NMS): {ckpt_args.get('end2end', 'N/A')}")
        print(f"   Епохи (Epochs):             {ckpt_args.get('epochs', 'N/A')}")
        print(f"   Patience (Рання зупинка):   {ckpt_args.get('patience', 'N/A')}")
        print(f"   Оптимізатор (Optimizer):    {ckpt_args.get('optimizer', 'N/A')}")
        print(f"   Прямокутне навчання (Rect): {ckpt_args.get('rect', 'N/A')}")
        print(f"   Аугментація Mosaic:         {ckpt_args.get('mosaic', 'N/A')}")
    print("="*50 + "\n")
    
    # Визначення порогів для поточного тесту
    test_conf = 0.25 if "with-nms" in SELECTED_EXPERIMENT else 0.05
    test_iou = 0.45 if "with-nms" in SELECTED_EXPERIMENT else 0.90
    print(f"⚙️ Запуск тестування з порогами: Conf={test_conf}, IoU (для NMS)={test_iou}")
    
    # Крок 4. Запуск тестування на тестовому спліті (split='test')
    test_results_dir = f"/kaggle/working/runs/detect/{SELECTED_EXPERIMENT}_test_evaluation"
    test_metrics = eval_model.val(
        data=yaml_path_baseline,
        split='test',  # Оцінюємо виключно на тестовій вибірці
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else 'cpu',
        rect=True,
        project='/kaggle/working/runs/detect',
        name=f"{SELECTED_EXPERIMENT}_test_evaluation",
        end2end = end2end_model,
        exist_ok=True,
        conf=test_conf,
        iou=test_iou
    )
    
    # Крок 5. Створення детального текстового звіту точності (test_metrics_summary.txt)
    summary_file_path = os.path.join(test_results_dir, "test_metrics_summary.txt")
    
    # Пошук папок зображень та анотацій тестового спліту
    test_img_dir = os.path.join(WORKING_DATASET_ROOT, "test", "images")
    test_lbl_dir = os.path.join(WORKING_DATASET_ROOT, "test", "labels")
    if not os.path.exists(test_img_dir):
        test_img_dir = os.path.join(WORKING_DATASET_ROOT, "images", "test")
        test_lbl_dir = os.path.join(WORKING_DATASET_ROOT, "labels", "test")
        
    test_viz_dir = os.path.join(test_results_dir, "test_visualizations")
    os.makedirs(test_viz_dir, exist_ok=True)
    
    # Глобальні лічильники рамок
    total_green = 0  # Ground Truth (Істинні об'єкти)
    total_white = 0  # True Positives (Правильні передбачення)
    total_red = 0    # False Positives (Помилкові передбачення)
    
    if os.path.exists(test_img_dir):
        valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        test_images = [img for img in glob.glob(os.path.join(test_img_dir, "*.*")) 
                       if os.path.splitext(img)[1].lower() in valid_extensions]
        
        print(f"🎨 Початок аналізу та малювання {len(test_images)} тестових зображень...")
        
        for img_path in tqdm(test_images, desc="Обчислення рамок (Зелений = GT, Білий = OK, Червоний = ERR)"):
            img = cv2.imread(img_path)
            if img is None:
                continue
            h_img, w_img = img.shape[:2]
            img_stem = Path(img_path).stem
            
            # А. Зчитуємо та зберігаємо координати Ground Truth
            gt_boxes = []
            label_path = os.path.join(test_lbl_dir, f"{img_stem}.txt")
            if os.path.exists(label_path):
                with open(label_path, "r") as f_gt:
                    for line in f_gt:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            cls_id, xc, yc, wb, hb = map(float, parts)
                            x1 = int((xc - wb / 2) * w_img)
                            y1 = int((yc - hb / 2) * h_img)
                            x2 = int((xc + wb / 2) * w_img)
                            y2 = int((yc + hb / 2) * h_img)
                            gt_boxes.append({
                                'coords': [x1, y1, x2, y2],
                                'cls': int(cls_id)
                            })
                            
            # Б. Отримуємо передбачення моделі з порогами поточного тесту
            preds = eval_model.predict(
                img_path, 
                conf=test_conf, 
                iou=test_iou,
                imgsz=640, 
                verbose=False
            )[0]
            
            pred_boxes = []
            for box in preds.boxes:
                coords = box.xyxy[0].cpu().numpy().astype(int).tolist()
                conf_val = float(box.conf[0])
                cls_id = int(box.cls[0])
                pred_boxes.append({
                    'coords': coords,
                    'cls': cls_id,
                    'conf': conf_val
                })
                
            # В. Зіставлення передбачень з GT (Bipartite Greedy Matching)
            matched_gts = set()
            matched_preds = set()
            candidates = []
            
            for g_idx, gt in enumerate(gt_boxes):
                for p_idx, pr in enumerate(pred_boxes):
                    if gt['cls'] == pr['cls']:
                        iou_val = calculate_single_iou(gt['coords'], pr['coords'])
                        if iou_val >= 0.5:
                            candidates.append((g_idx, p_idx, iou_val))
                            
            candidates.sort(key=lambda x: x[2], reverse=True)
            for g_idx, p_idx, iou_val in candidates:
                if g_idx not in matched_gts and p_idx not in matched_preds:
                    matched_gts.add(g_idx)
                    matched_preds.add(p_idx)
                    
            # Г. Малювання рамок на кадрі
            # 1. Малюємо Ground Truth (Зелений колір)
            for gt in gt_boxes:
                x1, y1, x2, y2 = gt['coords']
                c_name = eval_model.names.get(gt['cls'], str(gt['cls']))
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, f"GT: {c_name}", (x1, max(15, y1 - 5)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
                total_green += 1
                
            # 2. Малюємо передбачення моделі (Білий або Червоний колір)
            for p_idx, pr in enumerate(pred_boxes):
                x1, y1, x2, y2 = pr['coords']
                c_name = eval_model.names.get(pr['cls'], str(pr['cls']))
                conf_val = pr['conf']
                
                if p_idx in matched_preds:
                    # Зараховано як правильне (Білий колір)
                    color = (255, 255, 255)
                    label = f"OK: {c_name} {conf_val:.2f}"
                    total_white += 1
                else:
                    # Помилка детекції (Червоний колір)
                    color = (0, 0, 255)
                    label = f"ERR: {c_name} {conf_val:.2f}"
                    total_red += 1
                    
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, label, (x1, min(h_img - 5, y2 + 12)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
                
            # Збереження готового кадру у папку тестування
            cv2.imwrite(os.path.join(test_viz_dir, f"visualized_{img_stem}.jpg"), img)
            
        print(f"✅ Усі анотовані зображення збережено в: {test_viz_dir}")
    else:
        print("[Попередження] Не вдалося знайти папку з тестовими зображеннями для малювання рамок.")

    # Створення текстового звіту точності (test_metrics_summary.txt)
    try:
        with open(summary_file_path, "w", encoding="utf-8") as f:
            f.write(f"=== ТЕКСТОВИЙ ЗВІТ ТОЧНОСТІ ТЕСТУВАННЯ (СПЛІТ: TEST) ===\n")
            f.write(f"Експеримент: {SELECTED_EXPERIMENT}\n")
            f.write(f"Дата оцінки: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Файл ваг: {resolved_weights_path}\n")
            f.write("-" * 60 + "\n\n")
            
            f.write("⚙️ Активні параметри інференсу моделі:\n")
            f.write(f"  Поріг Conf:                   {test_conf}\n")
            f.write(f"  Поріг IoU для NMS:            {test_iou}\n")
            f.write(f"  Режим End-to-End (Без NMS):   {end2end if end2end is not None else 'Default (False)'}\n\n")
            
            f.write("📈 Загальні метрики точності моделі (mAP):\n")
            f.write(f"  Точність (Precision):         {test_metrics.box.mp:.6f}\n")
            f.write(f"  Повнота (Recall):             {test_metrics.box.mr:.6f}\n")
            f.write(f"  mAP@50:                       {test_metrics.box.map50:.6f}\n")
            f.write(f"  mAP@50-95:                    {test_metrics.box.map:.6f}\n\n")
            
            f.write("📊 Статистика згенерованих рамок на тестовій вибірці:\n")
            f.write(f"  Кількість істинних об'єктів (Зелені / GT):          {total_green}\n")
            f.write(f"  Кількість зарахованих правильних детекцій (Білі):    {total_white}\n")
            f.write(f"  Кількість помилкових детекцій (Червоні):             {total_red}\n")
            f.write(f"  Загальна кількість передбачень моделі:              {total_white + total_red}\n\n")
            
            f.write("⏱️ Швидкість обробки зображень (в мілісекундах на кадр):\n")
            for speed_key, speed_val in test_metrics.speed.items():
                f.write(f"  {speed_key}: {speed_val:.3f} ms\n")
            f.write("\n")
            
            f.write("📊 Детальні метрики точності по класах:\n")
            class_names = eval_model.names
            ap_class_indices = getattr(test_metrics.box, 'ap_class_index', [])
            
            if len(ap_class_indices) > 0:
                for idx, class_idx in enumerate(ap_class_indices):
                    c_name = class_names.get(class_idx, f"ID_{class_idx}")
                    p_class = test_metrics.box.p[idx] if len(test_metrics.box.p) > idx else 0.0
                    r_class = test_metrics.box.r[idx] if len(test_metrics.box.r) > idx else 0.0
                    ap50_class = test_metrics.box.ap50[idx] if len(test_metrics.box.ap50) > idx else 0.0
                    ap95_class = test_metrics.box.ap[idx] if len(test_metrics.box.ap) > idx else 0.0
                    
                    f.write(f"  Клас [{class_idx}] '{c_name}':\n")
                    f.write(f"    Precision: {p_class:.6f}\n")
                    f.write(f"    Recall:    {r_class:.6f}\n")
                    f.write(f"    mAP@50:    {ap50_class:.6f}\n")
                    f.write(f"    mAP@50-95: {ap95_class:.6f}\n")
            else:
                f.write("  Дані по окремих класах відсутні.\n")
                
        print(f"📄 Згенеровано та збережено текстовий звіт: {summary_file_path}")
    except Exception as e:
        print(f"[Попередження] Помилка створення текстового звіту точності: {e}")

    # Крок 6. Консольний вивід результатів та кількості рамок
    print("\n" + "="*50)
    print(f"     РЕЗУЛЬТАТИ ТЕСТУВАННЯ НА СПЛІТІ 'TEST' ({SELECTED_EXPERIMENT})")
    print("="*50)
    print(f" Точність (Precision):         {test_metrics.box.mp:.4f}")
    print(f" Повнота (Recall):             {test_metrics.box.mr:.4f}")
    print(f" mAP@50 (mAP):                 {test_metrics.box.map50:.4f}")
    print(f" mAP@50-95:                    {test_metrics.box.map:.4f}")
    print("-" * 50)
    print(f" 🟢 Зелені рамки (Ground Truth):        {total_green}")
    print(f" ⚪ Білі рамки (Правильні / OK):        {total_white}")
    print(f" 🔴 Червоні рамки (Помилкові / ERR):     {total_red}")
    print(f" 📦 Загальна к-сть передбачень моделі:   {total_white + total_red}")
    print("="*50 + "\n")
    
    # Крок 7. Створення фінального ZIP-архіву тестування
    if os.path.exists(test_results_dir):
        archive_name = f"/kaggle/working/{SELECTED_EXPERIMENT}_test_evaluation"
        shutil.make_archive(archive_name, 'zip', test_results_dir)
        zip_file_eval = f"{archive_name}.zip"
        size_mb = os.path.getsize(zip_file_eval) / (1024 * 1024)
        print(f"✅ Створено повний ZIP-архів тестування: {zip_file_eval} ({size_mb:.2f} MB)")
        
        # Завантаження архівованих тестів на Hugging Face
        if active_token and active_token != "ВСТАВТЕ_HF_WRITE_TOKEN_ТУТ":
            try:
                print(f"📤 Відправка архіву на Hugging Face...")
                api = HfApi(token=active_token)
                api.upload_file(
                    path_or_fileobj=zip_file_eval,
                    path_in_repo=f"{SELECTED_EXPERIMENT}_test_evaluation.zip",
                    repo_id=selected_repo_id,
                    commit_message=f"Upload evaluation with config parameters, OK (White) and ERR (Red) bounding boxes for {SELECTED_EXPERIMENT}"
                )
                print("🚀 Архів тестування успішно завантажено на Hugging Face.")
            except Exception as e:
                print(f"❌ Не вдалося відправити тест-архів на Hugging Face: {e}")
else:
    print(f"❌ Не вдалося знайти або імпортувати ваги моделі для експерименту: {SELECTED_EXPERIMENT}")

In [ ]:
# Клітинка 8: Глибока автономна діагностика детекції на основі ваг з Hugging Face
import os
import cv2
import glob
import shutil
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou

# ==================== НАЛАШТУВАННЯ ДІАГНОСТИКИ ====================
DIAG_EXPERIMENT = "yolov26_with_nms"  # Оберіть експеримент для діагностики
DATASET_SPLIT = "test+val"            # Спліти для глибокого аналізу
CONF_THRESH = 0.25
IOU_THRESH = 0.3
# ==================================================================

diag_repo_id = f"ituvtu/{DIAG_EXPERIMENT.replace('_', '-')}"
resolved_diag_weights = resolve_model_weights(DIAG_EXPERIMENT, diag_repo_id, token=HF_TOKEN)

if resolved_diag_weights and os.path.exists(resolved_diag_weights):
    diag_model = YOLO(resolved_diag_weights)
    
    # Підготовка робочих папок
    diagnostic_root = "/kaggle/working/diagnostic_analysis"
    dir_plots = os.path.join(diagnostic_root, "plots")
    dir_tp = os.path.join(diagnostic_root, "annotated_images/true_positives")
    dir_fp = os.path.join(diagnostic_root, "annotated_images/false_positives")
    dir_preds = os.path.join(diagnostic_root, "predictions_yolo")
    
    if os.path.exists(diagnostic_root):
        shutil.rmtree(diagnostic_root)
        
    os.makedirs(dir_plots, exist_ok=True)
    os.makedirs(dir_tp, exist_ok=True)
    os.makedirs(dir_fp, exist_ok=True)
    os.makedirs(dir_preds, exist_ok=True)
    
    splits_to_process = [s for s in ["val", "test"] if s in DATASET_SPLIT]
    image_data_list = []
    
    # Збір зображень
    for split in splits_to_process:
        img_dir = os.path.join(WORKING_DATASET_ROOT, "images", split)
        lbl_dir = os.path.join(WORKING_DATASET_ROOT, "labels", split)
        if os.path.exists(img_dir):
            for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp"]:
                for img_p in glob.glob(os.path.join(img_dir, ext)):
                    image_data_list.append((img_p, lbl_dir))
                    
    print(f"🔍 Знайдено {len(image_data_list)} зображень для діагностичного аналізу.")
    
    detected_centers_x, detected_centers_y = [], []
    missed_centers_x, missed_centers_y = [], []
    successful_files, error_files = [], []
    
    for img_path, lbl_dir in tqdm(image_data_list, desc="Обчислення координат та детекція"):
        img_name = Path(img_path).name
        img_stem = Path(img_path).stem
        label_path = os.path.join(lbl_dir, f"{img_stem}.txt")
        
        img = cv2.imread(img_path)
        if img is None:
            continue
        h_img, w_img = img.shape[:2]
        
        results = diag_model.predict(img_path, conf=CONF_THRESH, imgsz=640, verbose=False)[0]
        
        # Збереження передбачень
        pred_txt_path = os.path.join(dir_preds, f"{img_stem}.txt")
        with open(pred_txt_path, "w") as f_pred:
            for box in results.boxes:
                cls_id = int(box.cls[0])
                xywhn = box.xywhn[0].cpu().numpy()
                conf = float(box.conf[0])
                f_pred.write(f"{cls_id} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f} {conf:.4f}\n")
                
        pred_boxes_xyxy = results.boxes.xyxy.cpu().numpy().tolist() if len(results.boxes) > 0 else []
        pred_classes = results.boxes.cls.cpu().numpy().astype(int).tolist() if len(results.boxes) > 0 else []
        
        gt_boxes_xyxy, gt_classes, gt_normalized_centers = [], [], []
        if os.path.exists(label_path):
            with open(label_path, "r") as f_gt:
                for line in f_gt.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, x_c, y_c, w_b, h_b = map(float, parts)
                        x1 = (x_c - w_b / 2) * w_img
                        y1 = (y_c - h_b / 2) * h_img
                        x2 = (x_c + w_b / 2) * w_img
                        y2 = (y_c + h_b / 2) * h_img
                        gt_boxes_xyxy.append([x1, y1, x2, y2])
                        gt_classes.append(int(cls))
                        gt_normalized_centers.append([x_c, y_c])
                        
        matched_gts, matched_preds = set(), set()
        if len(gt_boxes_xyxy) > 0 and len(pred_boxes_xyxy) > 0:
            gt_tensor = torch.tensor(gt_boxes_xyxy, dtype=torch.float32)
            pred_tensor = torch.tensor(pred_boxes_xyxy, dtype=torch.float32)
            iou_matrix = box_iou(gt_tensor, pred_tensor).cpu().numpy()
            
            candidates = []
            for g_idx in range(len(gt_boxes_xyxy)):
                for p_idx in range(len(pred_boxes_xyxy)):
                    iou_val = float(iou_matrix[g_idx, p_idx])
                    if iou_val >= IOU_THRESH and gt_classes[g_idx] == pred_classes[p_idx]:
                        candidates.append((g_idx, p_idx, iou_val))
            candidates.sort(key=lambda x: x[2], reverse=True)
            
            for g_idx, p_idx, iou_val in candidates:
                if g_idx not in matched_gts and p_idx not in matched_preds:
                    matched_gts.add(g_idx)
                    matched_preds.add(p_idx)
                    detected_centers_x.append(gt_normalized_centers[g_idx][0])
                    detected_centers_y.append(gt_normalized_centers[g_idx][1])
                    
            for g_idx in range(len(gt_boxes_xyxy)):
                if g_idx not in matched_gts:
                    missed_centers_x.append(gt_normalized_centers[g_idx][0])
                    missed_centers_y.append(gt_normalized_centers[g_idx][1])
        else:
            for g_idx in range(len(gt_boxes_xyxy)):
                missed_centers_x.append(gt_normalized_centers[g_idx][0])
                missed_centers_y.append(gt_normalized_centers[g_idx][1])
                
        fn_count = len(gt_boxes_xyxy) - len(matched_gts)
        fp_count = len(pred_boxes_xyxy) - len(matched_preds)
        tp_count = len(matched_gts)
        
        if fn_count == 0 and fp_count == 0 and tp_count > 0:
            successful_files.append(img_name)
        else:
            error_files.append(img_name)
            
        # Візуалізація результатів
        if tp_count > 0:
            img_tp = img.copy()
            for g_idx in matched_gts:
                box = [int(c) for c in gt_boxes_xyxy[g_idx]]
                cv2.rectangle(img_tp, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
            cv2.imwrite(os.path.join(dir_tp, f"tp_{img_stem}.jpg"), img_tp)
            
        if fp_count > 0:
            img_fp = img.copy()
            for p_idx, pred_box in enumerate(pred_boxes_xyxy):
                if p_idx not in matched_preds:
                    box = [int(c) for c in pred_box]
                    cv2.rectangle(img_fp, (box[0], box[1]), (box[2], box[3]), (255, 0, 0), 2)
            cv2.imwrite(os.path.join(dir_fp, f"fp_{img_stem}.jpg"), img_fp)
            
    # Побудова та збереження карт щільності
    def save_density_plot(x, y, title, filename):
        plt.figure(figsize=(6, 6))
        if len(x) > 0:
            try:
                from scipy.stats import gaussian_kde
                xy = np.vstack([x, y])
                z = gaussian_kde(xy)(xy)
                idx = z.argsort()
                plt.scatter(np.array(x)[idx], np.array(y)[idx], c=z[idx], cmap='Blues', s=15, edgecolor='none')
            except Exception:
                plt.scatter(x, y, color='royalblue', s=15, alpha=0.5)
        plt.title(title)
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        plt.gca().set_aspect('equal', adjustable='box')
        plt.savefig(os.path.join(dir_plots, filename), dpi=150)
        plt.close()
        
    save_density_plot(detected_centers_x, detected_centers_y, "True Positives distribution", "tp_density.png")
    save_density_plot(missed_centers_x, missed_centers_y, "False Negatives distribution", "fn_density.png")
    
    # Архівування діагностики
    diag_zip = f"/kaggle/working/{DIAG_EXPERIMENT}_diagnostic_results"
    shutil.make_archive(diag_zip, 'zip', diagnostic_root)
    print(f"📦 Діагностику завершено. Результати запаковано у: {diag_zip}.zip")
else:
    print("❌ Не вдалося виконати діагностику: модель відсутня.")

In [ ]:
# Клітинка 9: Аналіз та локалізація критичних помилок детекції
import os
import cv2
import glob
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO

# Вибір моделі для аналізу
ANALYZE_EXPERIMENT = "yolov26_with_nms"
resolved_weights = resolve_model_weights(ANALYZE_EXPERIMENT, f"ituvtu/{ANALYZE_EXPERIMENT.replace('_', '-')}", token=HF_TOKEN)

images_test_dir = os.path.join(WORKING_DATASET_ROOT, "images/test")
labels_test_dir = os.path.join(WORKING_DATASET_ROOT, "labels/test")
error_output_dir = f"/kaggle/working/{ANALYZE_EXPERIMENT}_worst_errors_analysis"

if resolved_weights and os.path.exists(resolved_weights) and os.path.exists(images_test_dir):
    os.makedirs(error_output_dir, exist_ok=True)
    diagnostic_model = YOLO(resolved_weights)
    img_files = sorted(glob.glob(os.path.join(images_test_dir, "*.*")))
    
    analyzed_errors = []
    
    for path in tqdm(img_files, desc="Пошук дефектів та помилок на тестах"):
        stem_name = Path(path).stem
        lbl_path = os.path.join(labels_test_dir, f"{stem_name}.txt")
        
        preds = diagnostic_model.predict(path, conf=0.25, imgsz=640, verbose=False)[0]
        h_orig, w_orig = preds.orig_shape
        
        pred_coords = [b.xyxy[0].cpu().numpy().astype(int).tolist() for b in preds.boxes]
        gt_coords = []
        
        if os.path.exists(lbl_path):
            with open(lbl_path, "r") as f:
                for line in f.readlines():
                    elements = line.strip().split()
                    if len(elements) == 5:
                        _, xc, yc, wb, hb = map(float, elements)
                        x1 = int((xc - wb/2) * w_orig)
                        y1 = int((yc - hb/2) * h_orig)
                        x2 = int((xc + wb/2) * w_orig)
                        y2 = int((yc + hb/2) * h_orig)
                        gt_coords.append([x1, y1, x2, y2])
                        
        matched_gts, matched_preds = set(), set()
        for g_idx, gt in enumerate(gt_coords):
            for p_idx, pr in enumerate(pred_coords):
                # Обчислення базового IoU перекриття
                x_l, y_t = max(gt[0], pr[0]), max(gt[1], pr[1])
                x_r, y_b = min(gt[2], pr[2]), min(gt[3], pr[3])
                if x_r >= x_l and y_b >= y_t:
                    intersection = (x_r - x_l) * (y_b - y_t)
                    area1 = (gt[2] - gt[0]) * (gt[3] - gt[1])
                    area2 = (pr[2] - pr[0]) * (pr[3] - pr[1])
                    if intersection / float(area1 + area2 - intersection) >= 0.3:
                        matched_gts.add(g_idx)
                        matched_preds.add(p_idx)
                        
        fn_count = len(gt_coords) - len(matched_gts)
        fp_count = len(pred_coords) - len(matched_preds)
        total_err = fn_count + fp_count
        
        if total_err > 0:
            analyzed_errors.append({
                'path': path, 'gt': gt_coords, 'pred': pred_coords,
                'fn': fn_count, 'fp': fp_count, 'err_sum': total_err
            })
            
    if analyzed_errors:
        df = pd.DataFrame(analyzed_errors).sort_values(by='err_sum', ascending=False)
        top_errors = df.head(10)
        
        for index, row in top_errors.iterrows():
            canvas = cv2.imread(row['path'])
            # Зелені рамки - Істинна розмітка (Ground Truth)
            for box in row['gt']:
                cv2.rectangle(canvas, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
            # Червоні рамки - Помилки передбачення моделі
            for box in row['pred']:
                cv2.rectangle(canvas, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
                
            out_filename = os.path.basename(row['path'])
            cv2.imwrite(os.path.join(error_output_dir, f"err_{row['err_sum']}_{out_filename}"), canvas)
            
        shutil.make_archive(error_output_dir, 'zip', error_output_dir)
        print(f"✅ Аналіз завершено. Текстури критичних помилок збережено у: {error_output_dir}.zip")
    else:
        print("Помилок детекції на тестовій вибірці не виявлено.")
else:
    print("[Попередження] Не знайдено ваги моделі або тестову вибірку.")

In [ ]:
# Клітинка 10: Тест продуктивності, швидкості та затримки (Inference Benchmark)
import os
import time
import torch
from ultralytics import YOLO

BENCHMARK_EXPERIMENT = "yolov26_with_nms"
resolved_bench_weights = resolve_model_weights(BENCHMARK_EXPERIMENT, f"ituvtu/{BENCHMARK_EXPERIMENT.replace('_', '-')}", token=HF_TOKEN)
computing_device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

if resolved_bench_weights and os.path.exists(resolved_bench_weights):
    print(f"Завантаження моделі на пристрій: {computing_device}")
    model_bench = YOLO(resolved_bench_weights)
    core_model = model_bench.model.to(computing_device)
    
    use_half = (computing_device == 'cuda:0')
    core_model.half() if use_half else core_model.float()
    core_model.eval()
    
    # Симуляція відеопотоку 640x384 (стандартний для FPV-дронів формат кадру)
    dummy_input = torch.randn(1, 3, 384, 640, device=computing_device)
    if use_half:
        dummy_input = dummy_input.half()
        
    print("Прогрів GPU для стабілізації частот (Warm-up)...")
    for _ in range(30):
        with torch.inference_mode():
            _ = core_model(dummy_input)
            
    if computing_device == 'cuda:0':
        torch.cuda.synchronize()
        
    iterations = 200
    print(f"Замір продуктивності ({iterations} ітерацій)...")
    latencies = []
    
    if computing_device == 'cuda:0':
        start_evt = torch.cuda.Event(enable_timing=True)
        end_evt = torch.cuda.Event(enable_timing=True)
        for _ in range(iterations):
            start_evt.record()
            with torch.inference_mode():
                _ = core_model(dummy_input)
            end_evt.record()
            torch.cuda.synchronize()
            latencies.append(start_evt.elapsed_time(end_evt))
    else:
        for _ in range(iterations):
            t_start = time.perf_counter()
            with torch.inference_mode():
                _ = core_model(dummy_input)
            t_end = time.perf_counter()
            latencies.append((t_end - t_start) * 1000)
            
    mean_latency = sum(latencies) / len(latencies)
    frames_per_second = 1000.0 / mean_latency
    
    print("\n" + "="*50)
    print("        МЕТРИКИ ШВИДКОДІЇ ТА ПРОПУСКНОЇ ЗДАТНОСТІ")
    print("="*50)
    print(f" Модель:              {os.path.basename(resolved_bench_weights)}")
    print(f" Обчислювач:          {torch.cuda.get_device_name(0) if computing_device == 'cuda:0' else 'CPU'}")
    print(f" Формат даних:        {'FP16 (Half)' if use_half else 'FP32 (Float)'}")
    print(f" Роздільна здатність: 640x384")
    print(f" Середня затримка:    {mean_latency:.2f} мс")
    print(f" Продуктивність:      {frames_per_second:.2f} FPS")
    print("="*50 + "\n")
else:
    print("[Попередження] Відсутні файли ваг для тесту швидкодії.")

In [ ]:
# Клітинка 11: Фінальна перевірка та вивід структури вихідних файлів
import os

print("Аналіз згенерованих та доступних для завантаження артефактів у /kaggle/working/:")
for root, dirs, files in os.walk("/kaggle/working"):
    level = root.replace("/kaggle/working", "").count(os.sep)
    indent = " " * 4 * level
    folder_name = os.path.basename(root)
    if folder_name and not folder_name.startswith("."):
        print(f"{indent}📁 {folder_name}/")
        sub_indent = " " * 4 * (level + 1)
        for f in files:
            if not f.startswith("."):
                file_path = os.path.join(root, f)
                size_mb = os.path.getsize(file_path) / (1024 * 1024)
                print(f"{sub_indent}📄 {f} ({size_mb:.2f} MB)")

## 6. Post-Training Quantization (TensorRT INT8 & QAT Pipeline)
Compilation for low-power edge accelerators (NVIDIA Jetson / Rockchip NPU) using TensorRT INT8 with automatic FP16 Fallback.

In [ ]:
!pip install --quiet --extra-index-url https://pypi.nvidia.com tensorrt-cu12 tensorrt-cu12-libs tensorrt-cu12-bindings

In [ ]:
# Клітинка 8: Оптимізована підготовка середовища та запуск PTQ (TensorRT INT8 & ONNX FP16)
import os
import sys
import time
import warnings
import threading
import contextlib
import io
import subprocess

# 1. Придушення попереджень Hugging Face та PyTorch
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Клас для красивого анімованого текстового індикатора (Spinner)
class SilentSpinner:
    def __init__(self, message="Обробка..."):
        self.message = message
        self.stop_running = False
        self.thread = None

    def _spin(self):
        spinner_cycle = ['⠋', '⠙', '⠹', '⠸', '⠼', '⠴', '⠦', '⠧', '⠇', '⠏']
        idx = 0
        while not self.stop_running:
            sys.stdout.write(f"\r{spinner_cycle[idx]} {self.message}")
            sys.stdout.flush()
            idx = (idx + 1) % len(spinner_cycle)
            time.sleep(0.1)
        sys.stdout.write("\r" + " " * (len(self.message) + 10) + "\r")
        sys.stdout.flush()

    def start(self):
        self.stop_running = False
        self.thread = threading.Thread(target=self._spin)
        self.thread.start()

    def stop(self, success=True, success_message="Успішно!"):
        self.stop_running = True
        if self.thread:
            self.thread.join()
        symbol = "✅" if success else "❌"
        print(f"{symbol} {success_message}")

# 2. Крок 1: Встановлення бібліотек у фоновому режимі (без зайвого сміття в консолі)
spinner = SilentSpinner("Встановлення та оптимізація системних залежностей (TensorRT, ModelOpt, ONNX)...")
spinner.start()

try:
    # Встановлюємо пакети одним тихим викликом, щоб Ultralytics не запускав шумний AutoUpdate
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
            "onnxslim>=0.1.82", 
            "tensorrt-cu12>=11.0.0", 
            "nvidia-modelopt[onnx]>=0.44"
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    spinner.stop(success=True, success_message="Усі необхідні бібліотеки оптимізації підготовлено.")
except Exception as e:
    spinner.stop(success=False, success_message=f"Виникла помилка під час налаштування бібліотек: {e}")

from ultralytics import YOLO

# 3. Крок 2: Завантаження ваг з Hugging Face
SELECTED_EXPERIMENT = "yolov26_musgd_e2e" 
SELECTED_REPO_ID = "ituvtu/yolov26-musgd-e2e"
active_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None
yaml_path_baseline = "/kaggle/working/data_yolov26_baseline.yaml"

spinner = SilentSpinner("Перевірка та імпорт моделі з Hugging Face...")
spinner.start()

resolved_weights_path = None
try:
    if 'resolve_model_weights' in locals() or 'resolve_model_weights' in globals():
        # Перехоплюємо стандартний вивід hf_hub_download, щоб прибрати сміття
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            resolved_weights_path = resolve_model_weights(
                experiment_name=SELECTED_EXPERIMENT,
                repo_id=SELECTED_REPO_ID,
                token=active_token
            )
    else:
        import glob
        found = glob.glob("/kaggle/working/**/best.pt", recursive=True)
        if found:
            resolved_weights_path = found[0]
            
    if resolved_weights_path and os.path.exists(resolved_weights_path):
        spinner.stop(success=True, success_message=f"Базову модель успішно завантажено: {resolved_weights_path}")
    else:
        raise FileNotFoundError()
except Exception:
    spinner.stop(success=False, success_message="Не вдалося імпортувати ваги моделі.")
    raise FileNotFoundError("Перевірте з'єднання з Hugging Face або наявність локального файлу 'best.pt'.")

# 4. Крок 3: Запуск PTQ квантизації (TensorRT INT8 з автоматичним FP16 Fallback)
print("\n" + "="*70)
print(" ЗАПУСК POST-TRAINING QUANTIZATION (PTQ) З АВТОМАТИЧНИМ КАЛІБРУВАННЯМ")
print("="*70)

ptq_model = YOLO(resolved_weights_path)
ptq_engine_path = None
log_buffer = io.StringIO()

spinner = SilentSpinner("Компіляція TensorRT INT8 моделі (триває калібрування)...")
spinner.start()

# Перенаправляємо весь вивід у буфер, щоб не засмічувати консоль
with contextlib.redirect_stdout(log_buffer), contextlib.redirect_stderr(log_buffer):
    try:
        ptq_engine_path = ptq_model.export(
            format="engine",
            int8=True,
            half=True,  # Дозволяє автоматичний FP16 Fallback для нестабільних шарів (виправляє Error Code 9)
            data=yaml_path_baseline,
            device=0,
            opset=12
        )
        export_success = True
    except Exception as e:
        export_success = False
        error_exception = e

if export_success and ptq_engine_path:
    spinner.stop(success=True, success_message=f"TensorRT INT8 модель успішно створена: {ptq_engine_path}")
else:
    spinner.stop(success=False, success_message="Помилка компіляції TensorRT. Дивіться деталі нижче:")
    print("\n" + "!"*70)
    print("                      КРИТИЧНІ ЛОГИ ПОМИЛКИ TENSORRT")
    print("!"*70)
    # Показуємо лише кінець логу з описом помилки збірки тактики
    logs = log_buffer.getvalue().split('\n')
    relevant_logs = [line for line in logs if line.strip()]
    for line in relevant_logs[-25:]:  # Останні 25 важливих рядків
        print(line)
    print("!"*70 + "\n")

# 5. Крок 4: Запуск PTQ квантизації в ONNX FP16 (найбільш стабільна та сумісна з Jetson точка входу)
spinner = SilentSpinner("Експорт апаратно-незалежної ONNX FP16 моделі для Jetson Orin Nano...")
spinner.start()

onnx_log_buffer = io.StringIO()
with contextlib.redirect_stdout(onnx_log_buffer), contextlib.redirect_stderr(onnx_log_buffer):
    try:
        ptq_onnx_path = ptq_model.export(
            format="onnx",
            half=True,  # Експортуємо стабільну модель у напівточності (FP16)
            opset=12
        )
        onnx_success = True
    except Exception:
        onnx_success = False

if onnx_success and ptq_onnx_path:
    spinner.stop(success=True, success_message=f"ONNX FP16 модель успішно створена: {ptq_onnx_path}")
else:
    spinner.stop(success=False, success_message="Не вдалося зберегти ONNX FP16 модель.")

In [ ]:
# Клітинка 9: Тонке налаштування моделі за допомогою Quantization-Aware Training (QAT)
import torch
from ultralytics import YOLO

# 1. Функція підготовки моделі до QAT (інтеграція PyTorch Quantization в Ultralytics)
def enable_qat(trainer):
    import torch
    print("\n[QAT] Ініціалізація шарів фейкової квантизації перед тренуванням...")
    
    m = trainer.model
    
    # Блокуємо внутрішній метод збереження моделі Ultralytics, який ламає серіалізацію на фінальній епосі
    trainer.save_model = lambda *args, **kwargs: False
    print("[QAT] Автоматичне збереження моделі успішно заблоковано для усунення помилки серіалізації.")
    
    # Об'єднуємо шари Conv і BN (Convolution & BatchNorm) для стабільності
    for mod in m.modules():
        if hasattr(mod, 'conv') and hasattr(mod, 'bn'):
            try:
                torch.ao.quantization.fuse_modules(mod, ['conv', 'bn'], inplace=True)
            except Exception:
                pass
                
    # Застосовуємо конфігурацію 'qnnpack' (рекомендовано для мобільних/edge ARM процесорів)
    m.qconfig = torch.ao.quantization.get_default_qat_qconfig('qnnpack')
    
    try:
        # Впроваджуємо фейкові квантизатори у структуру графів моделі
        torch.ao.quantization.prepare_qat(m, inplace=True)
        print("[QAT] Модель успішно підготовлено до навчання в режимі INT8.")
    except Exception as e:
        print(f"[Попередження] Помилка підготовки PyTorch QAT: {e}")
        print("Спроба застосувати вибіркове налаштування для згорткових шарів...")
        for name, module in m.named_modules():
            if isinstance(module, torch.nn.Conv2d):
                module.qconfig = torch.ao.quantization.get_default_qat_qconfig('qnnpack')
        torch.ao.quantization.prepare_qat(m, inplace=True)
        
    m.train()

# 2. Функція фінального збереження ваг QAT у форматі state_dict (безпечно для pickle)
def convert_qat(trainer):
    trainer.model.eval()
    print("\n[QAT] Збереження навчених ваг QAT у форматі state_dict (без помилок серіалізації)...")
    try:
        # ВИПРАВЛЕНО: Змінено назву на yolov26_qat_int8_state.pth для повної сумісності з Клітинкою №11
        save_path = '/kaggle/working/yolov26_qat_int8_state.pth'
        torch.save(trainer.model.state_dict(), save_path)
        print(f"✅ Навчені ваги QAT успішно збережено: {save_path}")
    except Exception as e:
        print(f"❌ Помилка збереження ваг QAT: {e}")

# 3. Запуск донавчання QAT
print("\n" + "="*50)
print("2. ЗАПУСК QUANTIZATION-AWARE TRAINING (QAT)")
print("="*50)

qat_model = YOLO(resolved_weights_path)

# Додаємо наші кастомні події
qat_model.add_callback('on_train_start', enable_qat)
qat_model.add_callback('on_train_end', convert_qat)

# Запускаємо донавчання з перехопленням очікуваного виключення файлової системи
try:
    qat_model.train(
        data=yaml_path_baseline,
        epochs=2,               # 2 епохи для калібрування QAT без перенавчання
        imgsz=640,
        batch=16,               # Фіксований стабільний батч
        device=0 if torch.cuda.is_available() else 'cpu',
        project='/kaggle/working/runs/detect',
        name='yolov26_qat_training',
        optimizer="SGD",        
        lr0=0.001,              
        amp=False,              # Вимикаємо AMP для стабільності обсерверів QAT
        rect=True,              
        save=False,             # Проміжне збереження вимкнено
        
        # Вимикаємо важкі аугментації для точного зняття статистики обсерверами
        mosaic=0.0,
        mixup=0.0,
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        fliplr=0.0,
        flipud=0.0,
        scale=0.0,
        translate=0.0
    )
except FileNotFoundError as e:
    # Перехоплюємо очікувану помилку перевірки диска в Ultralytics
    if "Training completed but no checkpoint was saved" in str(e):
        print("\n" + "="*60)
        print("✅ QAT НАВЧАННЯ ЗАВЕРШЕНО УСПІШНО!")
        print("   (Обхідна система успішно запобігла помилці серіалізації)")
        print("="*60)
    else:
        raise e

# 4. Експорт QAT моделі у TensorRT Engine та ONNX прямо з оперативної пам'яті
print("\n[QAT] Експорт оптимізованої QAT-моделі прямо з оперативної пам'яті...")

# Конвертуємо IterableSimpleNamespace назад у класичні словники Python для сумісності з експортером
for obj in [qat_model, qat_model.model]:
    if hasattr(obj, 'args') and obj.args is not None:
        try:
            obj.args = dict(obj.args)
        except Exception:
            pass

# --- НОВИЙ КРОК: Конвертація моделі в Reference Quantized перед експортом ---
if hasattr(qat_model, 'model') and qat_model.model is not None:
    print("[QAT] Переведення моделі в режим оцінки (eval)...")
    qat_model.model.eval()
    
    print("[QAT] Конвертація QAT-моделі до представлення Reference Quantized (QDQ)...")
    try:
        #is_reference=True дозволяє експортувати модель у форматі ONNX QDQ
        torch.ao.quantization.convert(qat_model.model, is_reference=True, inplace=True)
        print("✅ Модель успішно конвертовано в Reference-режим (QDQ).")
    except Exception as e:
        print(f"⚠️ Помилка під час конвертації моделі: {e}")

# ВИПРАВЛЕНО: Відновлюємо втрачені службові метадані моделі після її динамічного модифікування PyTorch QAT
if hasattr(qat_model, 'model') and qat_model.model is not None:
    print("[QAT] Відновлення службових метаданих (task, names, stride)...")
    qat_model.model.task = 'detect'  # Задача детекції
    
    if hasattr(qat_model, 'names') and qat_model.names is not None:
        qat_model.model.names = qat_model.names  # Імена класів
        
    if hasattr(qat_model, 'stride') and qat_model.stride is not None:
        qat_model.model.stride = qat_model.stride
    else:
        # Крок моделі (stride є обов'язковим для експорту)
        qat_model.model.stride = torch.tensor([8.0, 16.0, 32.0]).to(device=0 if torch.cuda.is_available() else 'cpu')

try:
    qat_engine_path = qat_model.export(
        format="engine",
        int8=True,
        half=False,         # half та int8 є взаємовиключними для TensorRT
        data=yaml_path_baseline,
        device=0,
        opset=13            # ВИПРАВЛЕНО: Opset 13+ необхідний для коректного мапування QDQ-шарів
    )
    print(f"✅ TensorRT QAT модель успішно експортовано: {qat_engine_path}")
except Exception as e:
    print(f"❌ Помилка експорту TensorRT QAT: {e}")
    qat_engine_path = None

try:
    qat_onnx_path = qat_model.export(
        format="onnx",
        half=False,         # ВИПРАВЛЕНО: half=False, оскільки модель вже квантована (QDQ INT8)
        opset=13            # ВИПРАВЛЕНО: Opset 13+ для підтримки QDQ-шарів
    )
    print(f"✅ ONNX QAT модель успішно експортовано: {qat_onnx_path}")
except Exception as e:
    print(f"❌ Помилка експорту ONNX QAT: {e}")
    qat_onnx_path = None

In [ ]:
# Клітинка 10: Валідація результатів та побудова порівняльної таблиці
import os
import time
import torch
from ultralytics import YOLO

# 1. Функція вимірювання розміру файлу
def get_file_size_mb(filepath):
    if filepath and os.path.exists(filepath):
        return os.path.getsize(filepath) / (1024 * 1024)
    return 0.0

# 2. Визначення шляхів до скомпільованих файлів
fp32_weights = resolved_weights_path

# Шукаємо автоматично створені шляхи експортованих моделей PTQ (TensorRT)
ptq_engine = "/kaggle/working/runs/detect/yolov26_musgd_e2e/weights/best.engine"
if not os.path.exists(ptq_engine):
    import glob
    found = glob.glob("/kaggle/working/**/best.engine", recursive=True)
    if found:
        ptq_engine = found[0]

# Шукаємо автоматично створені шляхи експортованих моделей PTQ (ONNX FP16)
ptq_onnx = "/kaggle/working/runs/detect/yolov26_musgd_e2e/weights/best.onnx"
if not os.path.exists(ptq_onnx):
    import glob
    found = glob.glob("/kaggle/working/**/best.onnx", recursive=True)
    if found:
        ptq_onnx = found[0]

# Шлях до збережених ваг QAT
qat_state_path = "/kaggle/working/yolov26_qat_int8_state.pth"

# 3. Словник для збору метрик
benchmark_results = {}

print("\n" + "="*50)
print("3. ЗАПУСК ПОРІВНЯЛЬНОГО ТЕСТУВАННЯ НА СПЛІТІ 'VAL'")
print("="*50)

# --- А. Тестування Baseline (FP32) ---
if fp32_weights and os.path.exists(fp32_weights):
    print("\nВалідація моделі: Baseline (FP32)...")
    try:
        model = YOLO(fp32_weights)
        val_metrics = model.val(
            data=yaml_path_baseline,
            split='test',
            imgsz=640,
            batch=16,
            device=0,
            rect=True,
            verbose=False
        )
        benchmark_results["Baseline"] = {
            "Format": "PyTorch",
            "Precision": "FP32",
            "mAP50": val_metrics.box.map50,
            "mAP50-95": val_metrics.box.map,
            "Latency": val_metrics.speed.get('inference', 0.0) + val_metrics.speed.get('postprocess', 0.0),
            "Size": get_file_size_mb(fp32_weights)
        }
    except Exception as e:
        print(f"❌ Помилка тестування Baseline: {e}")

# --- Б. Тестування PyTorch (FP16) ---
if fp32_weights and os.path.exists(fp32_weights):
    print("\nВалідація моделі: PyTorch (FP16)...")
    try:
        model = YOLO(fp32_weights)
        val_metrics = model.val(
            data=yaml_path_baseline,
            split='test',
            imgsz=640,
            batch=16,
            device=0,
            rect=True,
            half=True,  # Вмикає нативні обчислення PyTorch у FP16 на GPU
            verbose=False
        )
        benchmark_results["PyTorch FP16"] = {
            "Format": "PyTorch",
            "Precision": "FP16",
            "mAP50": val_metrics.box.map50,
            "mAP50-95": val_metrics.box.map,
            "Latency": val_metrics.speed.get('inference', 0.0) + val_metrics.speed.get('postprocess', 0.0),
            "Size": get_file_size_mb(fp32_weights) / 2.0  # Теоретичний розмір збереженого FP16 файлу ваг
        }
    except Exception as e:
        print(f"❌ Помилка тестування PyTorch FP16: {e}")

# --- В. Тестування PTQ (ONNX FP16) ---
if ptq_onnx and os.path.exists(ptq_onnx):
    print("\nВалідація моделі: PTQ (ONNX FP16)...")
    try:
        model = YOLO(ptq_onnx)
        val_metrics = model.val(
            data=yaml_path_baseline,
            split='test',
            imgsz=640,
            batch=16,
            device=0,
            rect=True,
            verbose=False
        )
        benchmark_results["PTQ ONNX"] = {
            "Format": "ONNX",
            "Precision": "FP16",
            "mAP50": val_metrics.box.map50,
            "mAP50-95": val_metrics.box.map,
            "Latency": val_metrics.speed.get('inference', 0.0) + val_metrics.speed.get('postprocess', 0.0),
            "Size": get_file_size_mb(ptq_onnx)
        }
    except Exception as e:
        print(f"❌ Помилка тестування ONNX FP16: {e}")

# --- Г. Тестування PTQ (TRT Mixed Precision) ---
if ptq_engine and os.path.exists(ptq_engine):
    print("\nВалідація моделі: PTQ (TRT Mixed Precision)...")
    try:
        model = YOLO(ptq_engine)
        val_metrics = model.val(
            data=yaml_path_baseline,
            split='test',
            imgsz=640,
            batch=16,
            device=0,
            rect=True,
            verbose=False
        )
        benchmark_results["PTQ TRT"] = {
            "Format": "TensorRT",
            "Precision": "INT8 + FP32",
            "mAP50": val_metrics.box.map50,
            "mAP50-95": val_metrics.box.map,
            "Latency": val_metrics.speed.get('inference', 0.0) + val_metrics.speed.get('postprocess', 0.0),
            "Size": get_file_size_mb(ptq_engine)
        }
    except Exception as e:
        print(f"❌ Помилка тестування TRT Mixed Precision: {e}")

# --- Ґ. Тестування QAT (PyTorch Simulation) ---
if qat_state_path and os.path.exists(qat_state_path):
    print("\nВалідація моделі: QAT (PyTorch Simulation)...")
    try:
        # 1. Ініціалізуємо базову структуру моделі YOLO
        qat_model_eval = YOLO(resolved_weights_path)
        m = qat_model_eval.model
        
        m.train()
        
        # 2. Відтворюємо структуру QAT (fuse та prepare_qat)
        for mod in m.modules():
            if hasattr(mod, 'conv') and hasattr(mod, 'bn'):
                try:
                    torch.ao.quantization.fuse_modules(mod, ['conv', 'bn'], inplace=True)
                except Exception:
                    pass
                    
        m.qconfig = torch.ao.quantization.get_default_qat_qconfig('qnnpack')
        torch.ao.quantization.prepare_qat(m, inplace=True)
        
        # 3. Завантажуємо ваги QAT
        state_dict = torch.load(qat_state_path, map_location='cuda:0' if torch.cuda.is_available() else 'cpu')
        m.load_state_dict(state_dict)
        
        m.eval()
        
        val_metrics = qat_model_eval.val(
            data=yaml_path_baseline,
            split='val',
            imgsz=640,
            batch=16,
            device=0,
            rect=True,
            verbose=False
        )
        
        benchmark_results["QAT Sim"] = {
            "Format": "PyTorch",
            "Precision": "INT8 Sim",
            "mAP50": val_metrics.box.map50,
            "mAP50-95": val_metrics.box.map,
            "Latency": val_metrics.speed.get('inference', 0.0) + val_metrics.speed.get('postprocess', 0.0),
            "Size": get_file_size_mb(qat_state_path)
        }
        print("✅ QAT модель успішно реконструйовано та перевірено.")
    except Exception as e:
        print(f"❌ Помилка тестування QAT: {e}")

# 4. Виведення результатів порівняння у консоль (Форматована таблиця)
print("\n" + "="*115)
print(f"{'Конфігурація':<15} | {'Формат':<10} | {'Точність':<18} | {'mAP50':<8} | {'mAP50-95':<9} | {'Затримка (ms)':<14} | {'FPS':<8} | {'Розмір (MB)':<12}")
print("="*115)

for name, metrics in benchmark_results.items():
    latency = metrics['Latency']
    fps = 1000.0 / latency if latency > 0 else 0.0
    
    # Робимо позначку для швидкості QAT симуляції
    latency_str = f"{latency:.2f}"
    fps_str = f"{fps:.1f}"
    if name == "QAT Sim":
        latency_str += " *"
        fps_str += " *"
        
    print(f"{name:<15} | {metrics['Format']:<10} | {metrics['Precision']:<18} | {metrics['mAP50']:<8.4f} | {metrics['mAP50-95']:<9.4f} | {latency_str:<14} | {fps_str:<8} | {metrics['Size']:<12.2f}")
print("="*115)
print(" * Примітка: Для QAT Sim швидкість відображає роботу програмної симуляції fake_quantize.")
print("Аналіз завершено. Результати готові для використання у тексті бакалаврської роботи.")

In [ ]:
# Клітинка 11: Завантаження оптимізованої моделі PyTorch FP16 на Hugging Face Hub
import os
import sys
import warnings
import torch
from huggingface_hub import HfApi

# Придушення зайвих попереджень
warnings.filterwarnings("ignore", category=UserWarning)

# 1. Визначення токенів та параметрів репозиторію
SELECTED_REPO_ID = "ituvtu/yolov26-musgd-e2e" # Змініть на свій репозиторій за потреби
active_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None

if not active_token:
    print("❌ Помилка: Токен Hugging Face (HF_TOKEN) не знайдено в сесії.")
    print("Будь ласка, переконайтеся, що ви авторизовані через Kaggle Secrets або hf_login().")
else:
    print(f"🔑 Токен виявлено. Цільовий репозиторій: {SELECTED_REPO_ID}")

    # Шляхи до файлів
    fp32_weights = resolved_weights_path
    fp16_weights_path = "/kaggle/working/yolov26_pytorch_fp16.pt"

    print("\n" + "="*70)
    print(" ПІДГОТОВКА ТА КОНВЕРТАЦІЯ МОДЕЛІ У НАПІВТОЧНІСТЬ (FP16)")
    print("="*70)

    # Конвертуємо модель у FP16 та зберігаємо на диск перед завантаженням
    try:
        print("⏳ Завантаження та перетворення ваг контрольної точки у FP16...")
        ckpt = torch.load(fp32_weights, map_location='cpu', weights_only=False)
        
        # ВИПРАВЛЕНО: Динамічно шукаємо де зберігаються ваги — в 'model' чи в 'ema' (EMA ваги часто використовуються YOLO)
        converted = False
        for key in ['model', 'ema']:
            if key in ckpt and ckpt[key] is not None:
                print(f"   - Знайдено ваги у ключі '{key}', конвертуємо у FP16...")
                ckpt[key].half()
                converted = True
                
        if not converted:
            raise KeyError("У структурі файлу контрольної точки не знайдено валідних ключів 'model' або 'ema'.")
            
        # Очищуємо стан оптимізатора для зменшення фінального розміру файлу
        if 'optimizer' in ckpt:
            ckpt['optimizer'] = None
            
        torch.save(ckpt, fp16_weights_path)
        print(f"✅ Модель PyTorch FP16 успішно створено: {fp16_weights_path} ({os.path.getsize(fp16_weights_path)/(1024*1024):.2f} MB)")
    except Exception as e:
        print(f"❌ Помилка при створенні PyTorch FP16 моделі: {e}")
        fp16_weights_path = None

    if fp16_weights_path and os.path.exists(fp16_weights_path):
        # Мапа відповідності для завантаження лише одного файлу PyTorch FP16
        files_to_upload = {
            fp16_weights_path: "yolov26_pytorch_fp16.pt"
        }

        print("\n" + "="*70)
        print(" ЗАВАНТАЖЕННЯ МОДЕЛІ PYTORCH FP16 НА HUGGING FACE HUB")
        print("="*70)

        try:
            # Ініціалізація клієнта API Hugging Face
            api = HfApi(token=active_token)
            
            # Створення або перевірка існування репозиторію
            api.create_repo(repo_id=SELECTED_REPO_ID, repo_type="model", exist_ok=True)
            print(f"✅ Репозиторій {SELECTED_REPO_ID} підготовлено до завантаження.")
            
            uploaded_count = 0
            skipped_count = 0

            # Завантаження єдиного цільового файлу
            for local_path, repo_name in files_to_upload.items():
                if os.path.exists(local_path):
                    file_size_mb = os.path.getsize(local_path) / (1024 * 1024)
                    print(f"\n📤 Завантаження: {local_path.split('/')[-1]} -> {repo_name} ({file_size_mb:.2f} MB)...")
                    
                    api.upload_file(
                        path_or_fileobj=local_path,
                        path_in_repo=repo_name,
                        repo_id=SELECTED_REPO_ID,
                        commit_message=f"Upload optimized {repo_name} (Size: {file_size_mb:.2f} MB)"
                    )
                    print(f"   ✅ Успішно завантажено як '{repo_name}'")
                    uploaded_count += 1
                else:
                    print(f"⚠️ Файл не знайдено (пропуск): {local_path}")
                    skipped_count += 1

            print("\n" + "="*70)
            print("                        СТАТУС СИНХРОНІЗАЦІЇ")
            print("="*70)
            print(f" Успішно завантажено файлів: {uploaded_count}")
            print(f" Пропущено (не знайдено):     {skipped_count}")
            print(f" Посилання на ваш репозиторій: https://huggingface.co/{SELECTED_REPO_ID}")
            print("="*70 + "\n")

        except Exception as e:
            print(f"❌ Критична помилка під час роботи з Hugging Face API: {e}")
    else:
        print("❌ Помилка: Файл моделі FP16 не був створений, завантаження скасовано.")

## 7. Rigorous FP32 vs FP16 Comparative Validation (Test Split: 759 Frames)
Direct side-by-side benchmarking on the isolated test flight split. Verifies inference speedup (+27.4% FPS) and precision retention (mAP50 0.7563).

In [ ]:
# Клітинка: Порівняльне тестування моделей FP32 та FP16 на тестовому спліті (split='test')
import os
import glob
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

# 1. Автоматичне визначення шляху до файлу конфігурації YAML
yaml_paths = glob.glob("/kaggle/working/*.yaml")
yaml_path = yaml_paths[0] if yaml_paths else "/kaggle/working/data_yolov26_baseline.yaml"

print(f"📦 Використовується конфігурація датасету: {yaml_path}")

# 2. Завантаження ваг з Hugging Face
print("📥 Завантаження моделей з Hugging Face...")
try:
    # Завантаження FP32 моделі (best.pt з експерименту 4)
    fp32_weights = hf_hub_download(
        repo_id="ituvtu/yolov26-musgd-e2e-experiment4",
        filename="best.pt",
        local_dir="/kaggle/working/comparison_weights",
        local_dir_use_symlinks=False
    )
    # Завантаження FP16 моделі (yolov26_pytorch_fp16.pt)
    fp16_weights = hf_hub_download(
        repo_id="ituvtu/yolov26-musgd-e2e",
        filename="yolov26_pytorch_fp16.pt",
        local_dir="/kaggle/working/comparison_weights",
        local_dir_use_symlinks=False
    )
    print("✅ Ваги завантажено успішно.")
except Exception as e:
    print(f"❌ Помилка завантаження файлів ваг: {e}")
    raise e

# 3. Функція для отримання розміру файлу
def get_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

# 4. Тестування моделі FP32 (звичайна точність)
print("\n🔄 Оцінка моделі: FP32 Precision (split='test')...")
model_fp32 = YOLO(fp32_weights)
metrics_fp32 = model_fp32.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    rect=True,
    half=False,  # Примусовий режим FP32
    verbose=False
)

# 5. Тестування моделі FP16 (напівточність)
print("\n🔄 Оцінка моделі: FP16 Precision (split='test')...")
model_fp16 = YOLO(fp16_weights)
metrics_fp16 = model_fp16.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    rect=True,
    half=True,   # Примусовий режим FP16 на GPU
    verbose=False
)

# 6. Збір та розрахунок метрик
latency_fp32 = metrics_fp32.speed.get('inference', 0.0) + metrics_fp32.speed.get('postprocess', 0.0)
fps_fp32 = 1000.0 / latency_fp32 if latency_fp32 > 0 else 0.0

latency_fp16 = metrics_fp16.speed.get('inference', 0.0) + metrics_fp16.speed.get('postprocess', 0.0)
fps_fp16 = 1000.0 / latency_fp16 if latency_fp16 > 0 else 0.0

# 7. Виведення результатів порівняння
print("\n" + "="*110)
print(f"{'Модель':<25} | {'mAP50':<10} | {'mAP50-95':<10} | {'Затримка (мс)':<15} | {'FPS':<10} | {'Розмір файлу (MB)':<18}")
print("="*110)

print(f"{'yolov26_FP32 (best.pt)':<25} | "
      f"{metrics_fp32.box.map50:<10.4f} | "
      f"{metrics_fp32.box.map:<10.4f} | "
      f"{latency_fp32:<15.2f} | "
      f"{fps_fp32:<10.1f} | "
      f"{get_size_mb(fp32_weights):<18.2f}")

print(f"{'yolov26_FP16 (fp16.pt)':<25} | "
      f"{metrics_fp16.box.map50:<10.4f} | "
      f"{metrics_fp16.box.map:<10.4f} | "
      f"{latency_fp16:<15.2f} | "
      f"{fps_fp16:<10.1f} | "
      f"{get_size_mb(fp16_weights):<18.2f}")

print("="*110 + "\n")

In [7]:
# Клітинка: Коректне порівняння FP32 та FP16 для однієї моделі (Experiment 4)
import os
import glob
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

# 1. Шлях до YAML
yaml_paths = glob.glob("/kaggle/working/*.yaml")
yaml_path = yaml_paths[0] if yaml_paths else "/kaggle/working/data_yolov26_baseline.yaml"

# 2. Завантажуємо ОДНУ модель (Experiment 4)
weights_path = hf_hub_download(
    repo_id="ituvtu/yolov26-musgd-e2e-experiment4",
    filename="best.pt",
    local_dir="/kaggle/working/comparison_weights",
    local_dir_use_symlinks=False
)

model = YOLO(weights_path)

# 3. Тест у FP32
metrics_fp32 = model.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    rect=True,
    half=False, # FP32
    verbose=False
)
latency_fp32 = metrics_fp32.speed.get('inference', 0.0) + metrics_fp32.speed.get('postprocess', 0.0)

# 4. Тест у FP16 (нативна конвертація на GPU)
metrics_fp16 = model.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    rect=True,
    half=True,  # FP16
    verbose=False
)
latency_fp16 = metrics_fp16.speed.get('inference', 0.0) + metrics_fp16.speed.get('postprocess', 0.0)

# 5. Результати
print("\n" + "="*80)
print(f"{'Режим точності':<15} | {'mAP50':<10} | {'mAP50-95':<10} | {'Затримка (ms)':<15} | {'FPS':<10}")
print("="*80)
print(f"{'FP32':<15} | {metrics_fp32.box.map50:<10.4f} | {metrics_fp32.box.map:<10.4f} | {latency_fp32:<15.2f} | {1000.0/latency_fp32:<10.1f}")
print(f"{'FP16':<15} | {metrics_fp16.box.map50:<10.4f} | {metrics_fp16.box.map:<10.4f} | {latency_fp16:<15.2f} | {1000.0/latency_fp16:<10.1f}")
print("="*80)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 54.0±5.1 MB/s, size: 81.2 KB)
val: Scanning /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test/labels... 759 images, 95 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 395.9it/s 1.9s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 10.8it/s 4.5s0.2s
                   all        759        748       0.84      0.684      0.759      0.438
Speed: 0.8ms preprocess, 2.2ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-3

In [8]:
# Клітинка: Конвертація моделі Experiment 4 у FP16, правильне збереження та перевірка швидкодії
import os
import glob
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

# 1. Визначення шляхів
yaml_paths = glob.glob("/kaggle/working/*.yaml")
yaml_path = yaml_paths[0] if yaml_paths else "/kaggle/working/data_yolov26_baseline.yaml"
output_fp16_path = "/kaggle/working/yolov26_pytorch_fp16.pt"

print("📥 Завантаження оригінальних ваг FP32 (Experiment 4)...")
fp32_weights = hf_hub_download(
    repo_id="ituvtu/yolov26-musgd-e2e-experiment4",
    filename="best.pt",
    local_dir="/kaggle/working/temp_weights",
    local_dir_use_symlinks=False
)

# 2. Очищення та правильна низькорівнева конвертація в FP16
print("⏳ Конвертація структури ваг у FP16...")
try:
    ckpt = torch.load(fp32_weights, map_location='cpu', weights_only=False)
    
    # Конвертуємо параметри моделі у FP16
    converted = False
    for key in ['model', 'ema']:
        if key in ckpt and ckpt[key] is not None:
            ckpt[key].half()
            converted = True
            
    if not converted:
        raise KeyError("Не знайдено ключів ваг 'model' або 'ema' у структурі файлу.")
        
    # Видаляємо стан оптимізатора для мінімізації розміру файлу
    if 'optimizer' in ckpt:
        ckpt['optimizer'] = None
        
    # Зберігаємо фінальний оптимізований файл
    torch.save(ckpt, output_fp16_path)
    print(f"✅ Ваги моделі успішно збережено: {output_fp16_path} ({os.path.getsize(output_fp16_path)/(1024*1024):.2f} MB)")
except Exception as e:
    print(f"❌ Помилка під час конвертації чи збереження: {e}")

# 3. Валідація збереженого файлу з обов'язковим параметром half=True
if os.path.exists(output_fp16_path):
    print("\n🔄 Запуск тестової перевірки збереженого файлу FP16 (з half=True)...")
    model_fp16 = YOLO(output_fp16_path)
    
    metrics = model_fp16.val(
        data=yaml_path,
        split='test',
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else 'cpu',
        rect=True,
        half=True,   # <--- КРИТИЧНО для уникнення затримки та приведення типів
        verbose=False
    )
    
    latency = metrics.speed.get('inference', 0.0) + metrics.speed.get('postprocess', 0.0)
    print("\n" + "="*95)
    print("                 РЕЗУЛЬТАТИ ВАЛІДАЦІЇ ЗБЕРЕЖЕНОГО ФАЙЛУ FP16")
    print("="*95)
    print(f" Локальний шлях:    {output_fp16_path}")
    print(f" Точність mAP50:    {metrics.box.map50:.4f}")
    print(f" Точність mAP50-95: {metrics.box.map:.4f}")
    print(f" Середня затримка:  {latency:.2f} мс")
    print(f" Швидкість:         {1000.0/latency:.1f} FPS")
    print("="*95)

📥 Завантаження оригінальних ваг FP32 (Experiment 4)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


best.pt:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

⏳ Конвертація структури ваг у FP16...
✅ Ваги моделі успішно збережено: /kaggle/working/yolov26_pytorch_fp16.pt (5.16 MB)

🔄 Запуск тестової перевірки збереженого файлу FP16 (з half=True)...
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 10.0±6.5 MB/s, size: 80.7 KB)
val: Scanning /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test/labels... 759 images, 95 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 136.0it/s 5.6s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 11.1it/s 4.3s0.1s
                   all        7

In [9]:
# Клітинка: Завантаження моделі на Hugging Face, скачування та фінальна верифікація роботи
import os
import glob
import torch
from ultralytics import YOLO
from huggingface_hub import HfApi, hf_hub_download

# 1. Параметри та токен авторизації
SELECTED_REPO_ID = "ituvtu/yolov26-musgd-e2e-experiment4"
active_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None

local_fp16_path = "/kaggle/working/yolov26_pytorch_fp16.pt"
download_test_dir = "/kaggle/working/download_verification"

# Автоматичне визначення конфігурації YAML
yaml_paths = glob.glob("/kaggle/working/*.yaml")
yaml_path = yaml_paths[0] if yaml_paths else "/kaggle/working/data_yolov26_baseline.yaml"

if not active_token:
    print("❌ Помилка: Токен Hugging Face (HF_TOKEN) не знайдено в сесії.")
    print("Переконайтеся, що ви авторизовані через Kaggle Secrets або hf_login().")
elif not os.path.exists(local_fp16_path):
    print(f"❌ Локальний файл {local_fp16_path} не знайдено. Спочатку запустіть попередню клітинку.")
else:
    # --- КРОК 1: Завантаження файлу на Hugging Face Hub ---
    print("\n" + "="*80)
    print(" 1. ЗАВАНТАЖЕННЯ МОДЕЛІ FP16 НА HUGGING FACE HUB")
    print("="*80)
    try:
        api = HfApi(token=active_token)
        # Перевірка/створення репозиторію
        api.create_repo(repo_id=SELECTED_REPO_ID, repo_type="model", exist_ok=True)
        
        file_size_mb = os.path.getsize(local_fp16_path) / (1024 * 1024)
        print(f"⏳ Відправка файлу: {local_fp16_path} -> {SELECTED_REPO_ID}/yolov26_pytorch_fp16.pt ({file_size_mb:.2f} MB)...")
        
        api.upload_file(
            path_or_fileobj=local_fp16_path,
            path_in_repo="yolov26_pytorch_fp16.pt",
            repo_id=SELECTED_REPO_ID,
            commit_message=f"Upload verified yolov26_pytorch_fp16.pt (Size: {file_size_mb:.2f} MB)"
        )
        print("✅ Файл успішно завантажено на Hugging Face Hub.")
    except Exception as e:
        print(f"❌ Сталася помилка під час надсилання на HF: {e}")

    # --- КРОК 2: Чисте скачування файлу для тесту ---
    print("\n" + "="*80)
    print(" 2. СКАЧУВАННЯ МОДЕЛІ З HF ДЛЯ ВЕРИФІКАЦІЇ")
    print("="*80)
    os.makedirs(download_test_dir, exist_ok=True)
    
    downloaded_weights = None
    try:
        print("⏳ Скачування свіжозавантаженої моделі у чисту папку...")
        downloaded_weights = hf_hub_download(
            repo_id=SELECTED_REPO_ID,
            filename="yolov26_pytorch_fp16.pt",
            token=active_token,
            local_dir=download_test_dir,
            local_dir_use_symlinks=False
        )
        print(f"✅ Файл успішно скачано: {downloaded_weights}")
    except Exception as e:
        print(f"❌ Не вдалося скачати файл з HF: {e}")

    # --- КРОК 3: Фінальне тестування швидкодії скачаного файлу ---
    if downloaded_weights and os.path.exists(downloaded_weights):
        print("\n" + "="*80)
        print(" 3. ФІНАЛЬНИЙ ТЕСТ ШВИДКОДІЇ ТА ТОЧНОСТІ ЗАВАНТАЖЕНИХ ВАГ")
        print("="*80)
        
        model_verification = YOLO(downloaded_weights)
        
        # Запуск тесту з обов'язковим параметром half=True
        metrics = model_verification.val(
            data=yaml_path,
            split='test',
            imgsz=640,
            batch=16,
            device=0,
            rect=True,
            half=True,   # <--- Гарантує високу швидкість без casting-затримки
            verbose=False
        )
        
        latency = metrics.speed.get('inference', 0.0) + metrics.speed.get('postprocess', 0.0)
        print("\n" + "="*85)
        print("                 РЕЗУЛЬТАТИ ВЕРИФІКАЦІЇ СКАЧАНОЇ З HF МОДЕЛІ FP16")
        print("="*85)
        print(f" Точність mAP50:    {metrics.box.map50:.4f} (Очікувано: 0.7563)")
        print(f" Точність mAP50-95: {metrics.box.map:.4f} (Очікувано: 0.4367)")
        print(f" Затримка:          {latency:.2f} мс")
        print(f" Продуктивність:    {1000.0/latency:.1f} FPS")
        print("="*85 + "\n")


 1. ЗАВАНТАЖЕННЯ МОДЕЛІ FP16 НА HUGGING FACE HUB
⏳ Відправка файлу: /kaggle/working/yolov26_pytorch_fp16.pt -> ituvtu/yolov26-musgd-e2e-experiment4/yolov26_pytorch_fp16.pt (5.16 MB)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Файл успішно завантажено на Hugging Face Hub.

 2. СКАЧУВАННЯ МОДЕЛІ З HF ДЛЯ ВЕРИФІКАЦІЇ
⏳ Скачування свіжозавантаженої моделі у чисту папку...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


yolov26_pytorch_fp16.pt:   0%|          | 0.00/5.41M [00:00<?, ?B/s]

✅ Файл успішно скачано: /kaggle/working/download_verification/yolov26_pytorch_fp16.pt

 3. ФІНАЛЬНИЙ ТЕСТ ШВИДКОДІЇ ТА ТОЧНОСТІ ЗАВАНТАЖЕНИХ ВАГ
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 64.0±11.2 MB/s, size: 85.4 KB)
val: Scanning /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test/labels... 759 images, 95 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 429.8it/s 1.8s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 11.5it/s 4.2s0.1s
                   all        759        748      0.838      0.684      0.7

In [6]:
# Клітинка: Коректне порівняння FP32 та FP16 (завантаження обох моделей з HF та порівняння розмірів)
import os
import glob
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

# 1. Автоматичне визначення шляху до файлу конфігурації YAML
yaml_paths = glob.glob("/kaggle/working/*.yaml")
yaml_path = yaml_paths[0] if yaml_paths else "/kaggle/working/data_yolov26_baseline.yaml"

REPO_ID = "ituvtu/yolov26-musgd-e2e-experiment4"
active_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None

print(f"📦 Конфігурація датасету: {yaml_path}")
print("📥 Завантаження обох моделей з Hugging Face Hub...")

try:
    # Завантаження оригінальної моделі FP32 (best.pt)
    fp32_weights = hf_hub_download(
        repo_id=REPO_ID,
        filename="best.pt",
        token=active_token,
        local_dir="/kaggle/working/comparison_weights",
        local_dir_use_symlinks=False
    )
    # Завантаження оптимізованої моделі FP16 (yolov26_pytorch_fp16.pt)
    fp16_weights = hf_hub_download(
        repo_id=REPO_ID,
        filename="yolov26_pytorch_fp16.pt",
        token=active_token,
        local_dir="/kaggle/working/comparison_weights",
        local_dir_use_symlinks=False
    )
    print("✅ Обидва файли ваг успішно завантажено.")
except Exception as e:
    print(f"❌ Помилка завантаження моделей з Hugging Face: {e}")
    raise e

# 2. Функція обчислення розміру файлу
def get_file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

size_fp32 = get_file_size_mb(fp32_weights)
size_fp16 = get_file_size_mb(fp16_weights)

# 3. Тестування моделі FP32 (звичайна точність)
print("\n🔄 Запуск валідації моделі: FP32 Precision (split='test')...")
model_fp32 = YOLO(fp32_weights)
metrics_fp32 = model_fp32.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    rect=True,
    half=False,  # Оцінка у стандартній точності FP32
    verbose=False
)
latency_fp32 = metrics_fp32.speed.get('inference', 0.0) + metrics_fp32.speed.get('postprocess', 0.0)
fps_fp32 = 1000.0 / latency_fp32 if latency_fp32 > 0 else 0.0

# 4. Тестування моделі FP16 (напівточність)
print("\n🔄 Запуск валідації моделі: FP16 Precision (split='test')...")
model_fp16 = YOLO(fp16_weights)
metrics_fp16 = model_fp16.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    rect=True,
    half=True,   # Обов'язково half=True для оптимізації інференсу ваг FP16
    verbose=False
)
latency_fp16 = metrics_fp16.speed.get('inference', 0.0) + metrics_fp16.speed.get('postprocess', 0.0)
fps_fp16 = 1000.0 / latency_fp16 if latency_fp16 > 0 else 0.0

# 5. Виведення результатів порівняння у консоль (Форматована таблиця)
print("\n" + "="*115)
print(f"{'Конфігурація':<25} | {'mAP50':<10} | {'mAP50-95':<10} | {'Затримка (ms)':<15} | {'FPS':<10} | {'Розмір файлу (MB)':<18}")
print("="*115)

print(f"{'yolov26_FP32 (best.pt)':<25} | "
      f"{metrics_fp32.box.map50:<10.4f} | "
      f"{metrics_fp32.box.map:<10.4f} | "
      f"{latency_fp32:<15.2f} | "
      f"{fps_fp32:<10.1f} | "
      f"{size_fp32:<18.2f}")

print(f"{'yolov26_FP16 (fp16.pt)':<25} | "
      f"{metrics_fp16.box.map50:<10.4f} | "
      f"{metrics_fp16.box.map:<10.4f} | "
      f"{latency_fp16:<15.2f} | "
      f"{fps_fp16:<10.1f} | "
      f"{size_fp16:<18.2f}")
print("="*115 + "\n")

📦 Конфігурація датасету: /kaggle/working/data_yolov26_baseline.yaml
📥 Завантаження обох моделей з Hugging Face Hub...
✅ Обидва файли ваг успішно завантажено.

🔄 Запуск валідації моделі: FP32 Precision (split='test')...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 171.0±38.9 MB/s, size: 89.6 KB)
val: Scanning /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test/labels... 759 images, 95 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 963.2it/s 0.8s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 10.4it/s 4.6s0.1s
                   all        759        748       0.84      0.684      0.759      0.438
Speed: 0.9ms preprocess, 2.2ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /kaggle/working/runs/detect/val

In [5]:
# Клітинка: Коректне порівняння FP32 та FP16 (завантаження обох моделей з HF та порівняння розмірів)
import os
import glob
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

# 1. Автоматичне визначення шляху до файлу конфігурації YAML
yaml_paths = glob.glob("/kaggle/working/*.yaml")
yaml_path = yaml_paths[0] if yaml_paths else "/kaggle/working/data_yolov26_baseline.yaml"

REPO_ID = "ituvtu/yolov26-musgd-e2e-experiment4"
active_token = HF_TOKEN if 'HF_TOKEN' in locals() or 'HF_TOKEN' in globals() else None

print(f"📦 Конфігурація датасету: {yaml_path}")
print("📥 Завантаження обох моделей з Hugging Face Hub...")

try:
    # Завантаження оригінальної моделі FP32 (best.pt)
    fp32_weights = hf_hub_download(
        repo_id=REPO_ID,
        filename="best.pt",
        token=active_token,
        local_dir="/kaggle/working/comparison_weights",
        local_dir_use_symlinks=False
    )
    # Завантаження оптимізованої моделі FP16 (yolov26_pytorch_fp16.pt)
    fp16_weights = hf_hub_download(
        repo_id=REPO_ID,
        filename="yolov26_pytorch_fp16.pt",
        token=active_token,
        local_dir="/kaggle/working/comparison_weights",
        local_dir_use_symlinks=False
    )
    print("✅ Обидва файли ваг успішно завантажено.")
except Exception as e:
    print(f"❌ Помилка завантаження моделей з Hugging Face: {e}")
    raise e

# 2. Функція обчислення розміру файлу
def get_file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

size_fp32 = get_file_size_mb(fp32_weights)
size_fp16 = get_file_size_mb(fp16_weights)

# 3. Тестування моделі FP32 (звичайна точність)
print("\n🔄 Запуск валідації моделі: FP32 Precision (split='test')...")
model_fp32 = YOLO(fp32_weights)
metrics_fp32 = model_fp32.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    rect=True,
    half=False,  # Оцінка у стандартній точності FP32
    verbose=False
)
latency_fp32 = metrics_fp32.speed.get('inference', 0.0) + metrics_fp32.speed.get('postprocess', 0.0)
fps_fp32 = 1000.0 / latency_fp32 if latency_fp32 > 0 else 0.0

# 4. Тестування моделі FP16 (напівточність)
print("\n🔄 Запуск валідації моделі: FP16 Precision (split='test')...")
model_fp16 = YOLO(fp16_weights)
metrics_fp16 = model_fp16.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    rect=True,
    half=True,   # Обов'язково half=True для оптимізації інференсу ваг FP16
    verbose=False
)
latency_fp16 = metrics_fp16.speed.get('inference', 0.0) + metrics_fp16.speed.get('postprocess', 0.0)
fps_fp16 = 1000.0 / latency_fp16 if latency_fp16 > 0 else 0.0

# 5. Виведення результатів порівняння у консоль (Форматована таблиця з додаванням Precision та Recall)
print("\n" + "="*141)
print(f"{'Конфігурація':<25} | {'mAP50':<10} | {'mAP50-95':<10} | {'Precision':<10} | {'Recall':<10} | {'Затримка (ms)':<15} | {'FPS':<10} | {'Розмір файлу (MB)':<18}")
print("="*141)

print(f"{'yolov26_FP32 (best.pt)':<25} | "
      f"{metrics_fp32.box.map50:<10.4f} | "
      f"{metrics_fp32.box.map:<10.4f} | "
      f"{metrics_fp32.box.mp:<10.4f} | "
      f"{metrics_fp32.box.mr:<10.4f} | "
      f"{latency_fp32:<15.2f} | "
      f"{fps_fp32:<10.1f} | "
      f"{size_fp32:<18.2f}")

print(f"{'yolov26_FP16 (fp16.pt)':<25} | "
      f"{metrics_fp16.box.map50:<10.4f} | "
      f"{metrics_fp16.box.map:<10.4f} | "
      f"{metrics_fp16.box.mp:<10.4f} | "
      f"{metrics_fp16.box.mr:<10.4f} | "
      f"{latency_fp16:<15.2f} | "
      f"{fps_fp16:<10.1f} | "
      f"{size_fp16:<18.2f}")
print("="*141 + "\n")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📦 Конфігурація датасету: /kaggle/working/data_yolov26_baseline.yaml
📥 Завантаження обох моделей з Hugging Face Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


best.pt:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

yolov26_pytorch_fp16.pt:   0%|          | 0.00/5.41M [00:00<?, ?B/s]

✅ Обидва файли ваг успішно завантажено.

🔄 Запуск валідації моделі: FP32 Precision (split='test')...
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 23.9±1.7 MB/s, size: 81.2 KB)
val: Scanning /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test/labels... 759 images, 95 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 496.5it/s 1.5s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 8.2it/s 5.8s<0.2s
                   all        759        748       0.84      0.684      0.759      0.438
Speed: 0.8ms preprocess, 2.3ms 

In [8]:
import os
import glob
import gc
import torch
import numpy as np
from ultralytics import YOLO

# 1. Очищення пам'яті GPU перед тестом
gc.collect()
torch.cuda.empty_cache()

# Шлях до конфігурації
yaml_paths = glob.glob("/kaggle/working/*.yaml")
yaml_path = yaml_paths[0] if yaml_paths else "/kaggle/working/data_yolov26_baseline.yaml"
fp32_weights = "/kaggle/working/comparison_weights/best.pt"
fp16_weights = "/kaggle/working/comparison_weights/yolov26_pytorch_fp16.pt"

# Ініціалізація моделей
model_fp32 = YOLO(fp32_weights)
model_fp16 = YOLO(fp16_weights)
device = 0 if torch.cuda.is_available() else 'cpu'

# 2. Попередній розігрів (Warm-up) за допомогою офіційного API predict
if torch.cuda.is_available():
    print("🔥 Розігрів GPU для FP32 та FP16...")
    # Створюємо порожнє чорне зображення для розігріву
    dummy_img = np.zeros((640, 640, 3), dtype=np.uint8)
    
    # Розігрів FP32 шляху обчислень
    _ = model_fp32.predict(dummy_img, device=device, half=False, verbose=False)
    
    # Розігрів FP16 шляху обчислень (half=True автоматично конвертує модель і вхід)
    _ = model_fp16.predict(dummy_img, device=device, half=True, verbose=False)
    
    torch.cuda.synchronize()
    print("✅ Розігрів завершено. Починаємо валідацію...")

# 3. Тестування FP32 (оцінка швидкості вже на розігрітій системі)
print("\n🔄 Валідація FP32...")
metrics_fp32 = model_fp32.val(
    data=yaml_path, 
    split='test', 
    imgsz=640, 
    batch=16, 
    device=device, 
    rect=True, 
    half=False, 
    verbose=False
)
latency_fp32 = metrics_fp32.speed.get('inference', 0.0) + metrics_fp32.speed.get('postprocess', 0.0)
fps_fp32 = 1000.0 / latency_fp32 if latency_fp32 > 0 else 0.0

# 4. Тестування FP16 (оцінка швидкості вже на розігрітій системі)
print("\n🔄 Валідація FP16...")
metrics_fp16 = model_fp16.val(
    data=yaml_path, 
    split='test', 
    imgsz=640, 
    batch=16, 
    device=device, 
    rect=True, 
    half=True, 
    verbose=False
)
latency_fp16 = metrics_fp16.speed.get('inference', 0.0) + metrics_fp16.speed.get('postprocess', 0.0)
fps_fp16 = 1000.0 / latency_fp16 if latency_fp16 > 0 else 0.0

# 5. Виведення результатів
print("\n" + "="*141)
print(f"{'Конфігурація':<25} | {'mAP50':<10} | {'mAP50-95':<10} | {'Precision':<10} | {'Recall':<10} | {'Затримка (ms)':<15} | {'FPS':<10}")
print("="*141)
print(f"{'yolov26_FP32 (best.pt)':<25} | {metrics_fp32.box.map50:<10.4f} | {metrics_fp32.box.map:<10.4f} | {metrics_fp32.box.mp:<10.4f} | {metrics_fp32.box.mr:<10.4f} | {latency_fp32:<15.2f} | {fps_fp32:<10.1f}")
print(f"{'yolov26_FP16 (fp16.pt)':<25} | {metrics_fp16.box.map50:<10.4f} | {metrics_fp16.box.map:<10.4f} | {metrics_fp16.box.mp:<10.4f} | {metrics_fp16.box.mr:<10.4f} | {latency_fp16:<15.2f} | {fps_fp16:<10.1f}")
print("="*141 + "\n")

🔥 Розігрів GPU для FP32 та FP16...
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
✅ Розігрів завершено. Починаємо валідацію...

🔄 Валідація FP32...
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.3±0.5 ms, read: 186.1±67.0 MB/s, size: 91.4 KB)
val: Scanning /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test/labels... 759 images, 95 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 803.7it/s 0.9s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ituvtu/military-dataset/dataset_v25_640/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 10.6it/s 4.5s0.1s
                   